# 13 — Element IDs, cross-version deduplication and the three tables

Owner: Member 1. `parse_paper(pdir)` turns one paper folder (`data/raw/<yymm-id>/`) into its rows of the
three Milestone 2 tables, for **every version listed in `metadata.json`**. This notebook only defines
functions and offline tests: it reads a paper folder and writes nothing. `15_run_parsing.ipynb` runs it over
every paper on Drive, in parallel, and writes `parsed.json`, the logs and the Parquet tables.

## Interface with notebooks 10–12 (agreed contract)

A tree node is `{"kind": str, "content": str, "children": list[node]}`:

- `kind` is one of `document, chapter, section, subsection, subsubsection, paragraph, abstract, itemize, item,
  sentence, formula, figure` (a table counts as `figure`). `sentence`, `formula` and `figure` are the leaves.
- `content` is the cleaned title of an inner node (`document`: the paper title; `abstract`: `Abstract`;
  `itemize`: empty; `item`: its label, usually empty) and the cleaned text of a leaf. Block math is
  `\begin{equation}...\end{equation}`, inline math `$...$`.

| Notebook (owner) | Function | Returns |
|---|---|---|
| `10_parse_tree` (Member 2) | `find_root(vdir)` | `Path` of the root `.tex` (the file with `\documentclass`), or `None` |
| | `gather(vdir)` | the root's text, comments removed, `\input`/`\include` inlined recursively; `None` without a root |
| | `build_tree(text)` | the `document` node of one version |
| `11_normalize` (Member 4) | `expand_macros(text)`, `clean_latex(text)`, `normalize_math(text)` | `str` → `str`; `clean_latex` keeps the `\cite{...}` commands (`\citep`, `\citet`, ...) |
| | `split_sentences(text)` | `list[str]` |
| `12_bibtex` (Member 3) | `extract_bib(vdir, raw_text)` | `list[dict]` with keys `citation_key, entry_type, title, authors, year, doi, journal, url` |
| | `dedup_bib(by_version)` | `(entries, key_map)`: `by_version` is `{version: entries}`, `key_map` is `{old_key: chosen_key}` |
| | `rename_cites(text, key_map)` | `str` |
| | `bib_source(vdir, raw_text)` | `bib`, `bibitem`, `missing_bbl` (the `.bib` was not shipped, the references were only in the dropped `.bbl`) or `none` |

When one of these notebooks is in `src/`, it is loaded below and its functions are used. Until then the
**fallback** of section 2 stands in, function by function (checked by name), so the pipeline runs end to
end. `IMPLEMENTATION` shows which one is in use, and `PARSER_VERSION` changes with it, so `15` parses again
the papers done with an older implementation.

## Steps for one paper (`parse_paper`)

1. Every version `v1 … vN` of `metadata.json` (not the `vN/` folders: about 5% of the papers are PDF-only
   and have none): `raw = gather(vdir)`. No `vN/` folder or no `.tex` → `empty_source`; `.tex` files but no
   root → `no_root`. Both are recorded and yield no rows, as the statement asks.
2. Every version, on the **raw** text: `extract_bib(vdir, raw)` and `bib_source(vdir, raw)`.
3. Once per paper: `entries, key_map = dedup_bib(by_version)`.
4. Every version: `text = rename_cites(normalize_math(clean_latex(expand_macros(raw))), key_map)`, then
   `tree = build_tree(text)`. Citations are renamed **before** the full-text dedup, so a sentence that cites
   `a` in v1 and its duplicate `b` in v2 is still one element.
5. Element IDs and the dedup across versions (below).
6. Rows of `elements`, `hierarchy`, `bibtex`, and per version: status, seconds, `bib_source`, node count.

## Element IDs and deduplication

- `element_id` = `<folder-id>#<prefix><n>`, e.g. `2609-32834#s12`, with `n` counted per prefix in order of
  first appearance. Prefixes: `document` `doc`, `chapter` `ch`, `section` `sec`, `subsection` `ssec`,
  `subsubsection` `sssec`, `paragraph` `par`, `abstract` `abs`, `itemize` `list`, `item` `item`,
  `sentence` `s`, `formula` `eq`, `figure` `fig`. The document root is `<folder-id>#doc` in every version.
- **Leaf** (`sentence`, `formula`, `figure`): key `(kind, content)`. Identical cleaned content gets one ID in
  any version and under any parent; a sentence that occurs twice in one version is one element with two
  `hierarchy` rows.
- **Inner node**: key `(kind, content, parent element_id, k)`. The parent keeps apart two `Setup`
  subsections of different sections (with the title alone their children would be merged); `k`, the number
  of earlier siblings with the same kind and content under that parent in that version, keeps apart two
  untitled lists of one section.

## Tables (the statement's schema, no extra column)

| Table | Columns | One row per |
|---|---|---|
| `elements` | `element_id, arxiv_id, kind, content` | distinct element of the paper, over all its versions |
| `hierarchy` | `arxiv_id, version, element_id, parent_id` | node–parent edge of each version, in document order |
| `bibtex` | `arxiv_id, citation_key, entry_type, title, authors, year, doi, journal, url` | unified BibTeX entry of the paper |

`arxiv_id` is the plain ID (`2609.32834`, not the folder name). `parent_id` is never null: the document
root has an `elements` row but no `hierarchy` row (the statement's example lists no edge for it), so in each
version the only parent that is never a child is `<folder-id>#doc`.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import bisect
import posixpath
import unicodedata
from collections import Counter
from pathlib import PurePosixPath

PARSER_BASE = "13.1"   # bump when the output of parse_paper changes: 15 then parses every paper again

## 1. Load the notebooks of Members 2–4

Each notebook of the contract that exists in `src/` is loaded once (`%run -i`). Its functions replace the
fallback of the same name in the next section.

In [ ]:
TEAM_NOTEBOOKS = {   # notebook -> the functions it provides (contract above)
    "10_parse_tree": ("find_root", "gather", "build_tree"),
    "11_normalize": ("expand_macros", "clean_latex", "normalize_math", "split_sentences"),
    "12_bibtex": ("extract_bib", "dedup_bib", "rename_cites", "bib_source"),
}
TEAM_LOADED = globals().get("TEAM_LOADED", set())
for _notebook in TEAM_NOTEBOOKS:
    if _notebook not in TEAM_LOADED and Path(f"./{_notebook}.ipynb").is_file():
        get_ipython().run_line_magic("run", f"-i ./{_notebook}.ipynb")
        TEAM_LOADED.add(_notebook)

## 2. FALLBACK — replaced by notebooks 10/11/12 when present

A small, regex-based implementation of the contract, written to follow the statement so that the full run
works before the members' notebooks arrive. Every function here is named `_fb_<name>`; section 2e binds the
contract name to it only if no member's notebook defined that name.

**Shared helpers.** Files are decoded as UTF-8 (Latin-1 if that fails), with `\r\n` and old Mac `\r` line
ends turned into `\n` (with `\r` only, a `%` comment would run to the end of the file). The `.tex`/`.bib` files of a version
are read once and kept for the current paper only (`_version_files`), because each read is slow on Drive.
`strip_comments` removes `%` comments (not `\%`) and `comment` environments; a line holding only a comment
disappears with its line break, so it does not split a paragraph.

In [ ]:
def _read_text(path):
    """Text of a file, UTF-8 or else Latin-1, with \\n line ends (old Mac files end lines with \\r only,
    and a comment would otherwise run to the end of the file)."""
    data = Path(path).read_bytes()
    try:
        text = data.decode("utf-8")
    except UnicodeDecodeError:
        text = data.decode("latin-1")
    return text.replace("\r\n", "\n").replace("\r", "\n")


def _group_end(text, start):
    """Index just after the {...} or [...] group that opens at text[start]; -1 if it never closes."""
    opening = text[start]
    closing = "}" if opening == "{" else "]"
    depth, i, n = 0, start, len(text)
    while i < n:
        c = text[i]
        if c == "\\":
            i += 2
            continue
        if c == opening:
            depth += 1
        elif c == closing:
            depth -= 1
            if depth == 0:
                return i + 1
        i += 1
    return -1


def _read_group(text, i, opening="{"):
    """(content, end) of the group at text[i] (after spaces and at most one line break), else (None, i)."""
    j = i
    while j < len(text) and text[j] in " \t":
        j += 1
    if j < len(text) and text[j] == "\n":
        j += 1
        while j < len(text) and text[j] in " \t":
            j += 1
    if j < len(text) and text[j] == opening:
        end = _group_end(text, j)
        if end > 0:
            return text[j + 1:end - 1], end
    return None, i


def _replace_commands(text, names, n_args, keep=None, optional=True):
    """Replace \\name*[opt]{a1}...{an} with '' (keep=None) or with its argument number `keep` (1-based).

    A command missing one of its mandatory arguments is left unchanged.
    """
    pattern = re.compile(r"\\(?:" + "|".join(names) + r")(?![a-zA-Z])\*?")
    out, pos = [], 0
    for m in pattern.finditer(text):
        if m.start() < pos:
            continue
        i = m.end()
        if optional:
            _, i = _read_group(text, i, "[")
        args = []
        for _ in range(n_args):
            arg, j = _read_group(text, i)
            if arg is None:
                break
            args.append(arg)
            i = j
        if len(args) < n_args:
            continue
        out.append(text[pos:m.start()])
        out.append(args[keep - 1] if keep else "")
        pos = i
    out.append(text[pos:])
    return "".join(out)


_COMMENT_LINE = re.compile(r"^[ \t]*%[^\n]*\n", re.M)
_COMMENT = re.compile(r"(?<!\\)((?:\\\\)*)%[^\n]*")
_COMMENT_ENV = re.compile(r"\\begin\{comment\}.*?\\end\{comment\}", re.S)


def strip_comments(text):
    """Remove % comments (not \\%) and comment environments."""
    text = _COMMENT_LINE.sub("", text)
    text = _COMMENT.sub(r"\1", text)
    return _COMMENT_ENV.sub("", text)


_VERSION_CACHE = {}   # vdir -> {"files": {relative path: text}, "bib": {relative path: entries}}


def _version_files(vdir):
    """{relative posix path: text} of every .tex/.bib file of a version (read once per paper)."""
    key = str(vdir)
    if key not in _VERSION_CACHE:
        files = {}
        vdir = Path(vdir)
        if vdir.is_dir():
            for path in sorted(vdir.rglob("*")):
                if path.suffix.lower() in (".tex", ".bib") and path.is_file():
                    files[path.relative_to(vdir).as_posix()] = _read_text(path)
        _VERSION_CACHE[key] = {"files": files, "bib": {}}
    return _VERSION_CACHE[key]["files"]


def _fallback(function):
    function._m2_fallback = True
    return function


def _squash(text):
    return " ".join(text.split())

### 2a. Root and multi-file gathering (Member 2)

- **Root**: the `.tex` file whose document has `\documentclass` (or `\documentstyle`) outside comments and
  that no other file inlines: usually the file with `\documentclass` itself, but `main.tex` when it starts
  with `\include{preamble}` and `preamble.tex` holds `\documentclass` (2605.22101). With several candidates
  (e.g. `main.tex` and `rebuttal.tex`), the one whose document has `\begin{document}`, then a main-like name
  (`main`, `ms`, `paper`, ...), then the shallowest and the largest.
- **Gathering**: `\input{f}`, `\input f`, `\include{f}`, `\subfile{f}` and `\import{dir}{f}` /
  `\subimport{dir}{f}` are replaced by the file's text, recursively, after its comments were removed, so a
  commented-out `\input` is plain text and is not followed. Paths are tried relative to the root's folder,
  then the including file's folder, with and without `.tex`, then ignoring case. A missing file (a `.bbl` or
  a figure that M1 did not keep) becomes empty; a cycle is cut. Text after `\endinput` is dropped, and an
  inlined file with its own `\documentclass` and `\begin{document}` (the `subfiles` package) contributes only
  its document body.

In [ ]:
_DOCCLASS = re.compile(r"\\document(?:class|style)(?![a-zA-Z])")
_MAIN_NAMES = ("main", "ms", "paper", "article", "manuscript", "root", "arxiv")
_INPUT = re.compile(
    r"\\(?:input|include|subfile|subfileinclude)(?![a-zA-Z])\s*(?:\{([^{}]*)\}|([^\s{}\\%]+))"
    r"|\\(?:sub)?(?:import|inputfrom|includefrom)(?![a-zA-Z])\*?\s*\{([^{}]*)\}\s*\{([^{}]*)\}")
_ENDINPUT = re.compile(r"\\endinput(?![a-zA-Z])")
_DOCUMENT_BODY = re.compile(r"\\begin\s*\{document\}(.*?)(?:\\end\s*\{document\}|$)", re.S)
MAX_GATHER_CHARS = 20_000_000   # a version larger than this after inlining is cut (cycles, generated files)


def _resolve_tex(name, folders, tex_files, lower):
    """Relative path of the .tex file that \\input{name} means, or None."""
    name = name.strip().strip('"').replace("\\", "/")
    if not name:
        return None
    for folder in folders:
        for candidate in (name, name + ".tex"):
            path = posixpath.normpath(posixpath.join(folder, candidate))
            if path in tex_files:
                return path
            if path.lower() in lower:
                return lower[path.lower()]
    return None


def _input_target(match, folder, root_folder, tex_files, lower):
    if match.group(3) is not None:   # \import{dir}{file}: relative to dir
        name = posixpath.join(match.group(3), match.group(4))
    else:
        name = match.group(1) if match.group(1) is not None else match.group(2)
    return _resolve_tex(name, dict.fromkeys((root_folder, folder, "")), tex_files, lower)


@_fallback
def _fb_find_root(vdir):
    """Path of the root .tex file of a version, or None (see 2a)."""
    files = _version_files(vdir)
    cache = _VERSION_CACHE[str(vdir)]
    if "root" in cache:
        return cache["root"]
    texts = {name: strip_comments(text) for name, text in files.items() if name.lower().endswith(".tex")}
    with_class = {name for name, text in texts.items() if _DOCCLASS.search(text)}
    if not with_class:
        cache["root"] = None
        return None
    lower = {name.lower(): name for name in texts}
    inputs = {name: {target for m in _INPUT.finditer(text)
                     if (target := _input_target(m, posixpath.dirname(name), posixpath.dirname(name), texts, lower))
                     and target != name}
              for name, text in texts.items()}
    inlined_by = {}
    for name, targets in inputs.items():
        for target in targets:
            inlined_by.setdefault(target, set()).add(name)

    def reachable(name):
        seen, todo = set(), [name]
        while todo:
            current = todo.pop()
            if current not in seen:
                seen.add(current)
                todo.extend(inputs.get(current, ()))
        return seen

    # The root is a file that no other file inlines and whose document has \documentclass, itself or in an
    # inlined file (main.tex with \include{preamble} holding \documentclass).
    candidates = {}
    for name in texts:
        files_in = reachable(name)
        # inlined by a file outside its own document (a cycle back to itself does not count)
        if inlined_by.get(name, set()) - files_in:
            continue
        if files_in & with_class:
            candidates[name] = files_in
    if not candidates:   # every candidate inlined by another one (a cycle): the files with \documentclass
        candidates = {name: reachable(name) for name in with_class}

    def rank(name):
        has_body = any("\\begin{document}" in texts[f] for f in candidates[name])
        return (not has_body, PurePosixPath(name).stem.lower() not in _MAIN_NAMES, name.count("/"),
                -sum(len(texts[f]) for f in candidates[name]), name)
    cache["root"] = Path(vdir) / min(candidates, key=rank)
    return cache["root"]


@_fallback
def _fb_gather(vdir):
    """Root text with comments removed and every \\input/\\include inlined, recursively; None without a root."""
    root = find_root(vdir)
    if root is None:
        return None
    files = _version_files(vdir)
    tex_files = {name for name in files if name.lower().endswith(".tex")}
    lower = {name.lower(): name for name in tex_files}
    try:
        root_name = Path(root).resolve().relative_to(Path(vdir).resolve()).as_posix()
    except ValueError:
        root_name = Path(root).name
    if root_name not in files:
        return None
    root_folder = posixpath.dirname(root_name)
    size = [0]

    def expand(name, stack):
        text = strip_comments(files[name])
        end = _ENDINPUT.search(text)
        if end:
            text = text[:end.start()]
        if len(stack) > 1 and _DOCCLASS.search(text) and "\\begin{document}" in text:
            text = _DOCUMENT_BODY.search(text).group(1)   # a subfile: keep its document body only
        folder = posixpath.dirname(name)

        def inline(match):
            target = _input_target(match, folder, root_folder, tex_files, lower)
            if target is None or target in stack or len(stack) > 30 or size[0] > MAX_GATHER_CHARS:
                return ""
            inner = expand(target, stack + (target,))
            size[0] += len(inner)
            return "\n" + inner + "\n"

        return _INPUT.sub(inline, text)

    return expand(root_name, (root_name,))

### 2b. Macros, cleanup and math (Member 4)

- `expand_macros`: the document's own `\newcommand`, `\renewcommand`, `\providecommand`,
  `\DeclareRobustCommand`, `\def` (simple `#1#2` parameters), `\let` and `\DeclareMathOperator`, with arguments
  and an optional default for the first one. Definitions are removed and every use is replaced, a few passes
  for nested macros. Structural commands (`\section`, `\cite`, `\begin`, ...) are never redefined here.
- **Math is found like TeX reads it** (`_math_spans`), left to right: `\\` and `\$` are escapes, `$a$$b$`
  is two inline formulas, inline math cannot cross a blank line, and a `$` that is never closed stays a plain
  `$`. A regular expression got this wrong on real papers (`\\$x$` in a table, `$\approx$$128$`), which
  shifted every later formula; the scanner is shared by the three steps below.
- `clean_latex`: formatting-only commands dropped (`\centering`, `\midrule`, `\vspace{}`, `\vskip 1pt`,
  `\label{}`, font switches, `\makeatletter ... \makeatother`, ...), the placement option of floats
  (`[htpb]`), front matter (`\author`, `\affiliation`, `\thanks`, ...), formatting wrappers unwrapped
  (`\textbf{x}` → `x`, also when `x` holds math); outside math: accents to Unicode (`\'e` → `é`), `~` →
  space, ``` `` ''``` → `"`, `--`/`---` → `–`/`—`. `\cite`, `\ref`, `\url`, `\footnote`, section commands and
  environments are kept for the next steps.
- `normalize_math`: inline math `\(...\)` and `\begin{math}` → `$...$`; block math `$$...$$`, `\[...\]`,
  `equation(*)`, `displaymath` → `\begin{equation} ... \end{equation}`; `align`, `eqnarray`, `flalign`,
  `alignat` become `\begin{equation} \begin{aligned} ... \end{aligned} \end{equation}` and `gather`,
  `multline` `gathered` (one block format that still compiles). `\label`, `\tag`, `\nonumber` are dropped and
  spaces collapsed, so the same formula matches across versions.
- `split_sentences`: splits at a period followed by a space and a word that does not start in lower case. It
  never splits inside math, inside a command's braces (`\cite{a.b}`, `\footnote{...}`, `\url{...}`), inside a
  URL, after an abbreviation (`e.g.`, `i.e.`, `et al.`, `Fig.`, `Eq.`, `Sec.`, ...), after an initial
  (`J. Smith`) or in a decimal number. As the statement says, only periods end a sentence.

In [ ]:
_PROTECTED_MACROS = frozenset({
    "begin", "end", "item", "part", "chapter", "section", "subsection", "subsubsection", "paragraph",
    "subparagraph", "title", "author", "abstract", "maketitle", "appendix", "input", "include", "bibliography",
    "bibitem", "documentclass", "usepackage", "caption", "label", "ref", "eqref", "footnote", "url", "href",
    "cite", "citep", "citet", "nocite", "newcommand", "renewcommand", "def", "let", "emph", "textbf", "textit",
})
_DEFINITION = re.compile(
    r"\\(?:(?:re|provide)?newcommand|DeclareRobustCommand)(?![a-zA-Z])\*?"
    r"|\\DeclareMathOperator(?![a-zA-Z])\*?|\\g?def(?![a-zA-Z])|\\let(?![a-zA-Z])")
_MACRO_NAME = re.compile(r"\\([a-zA-Z@]+|.)")
_DEF_PARAMS = re.compile(r"(?:#\d)*")
MAX_MACRO_PASSES = 6


def _read_macro_name(text, i):
    """(name without backslash, end) of \\name or {\\name} at text[i], else (None, i)."""
    inner, end = _read_group(text, i)
    if inner is not None:
        m = _MACRO_NAME.fullmatch(inner.strip())
        return (m.group(1), end) if m else (None, i)
    j = i
    while j < len(text) and text[j] in " \t":
        j += 1
    m = _MACRO_NAME.match(text, j)
    return (m.group(1), m.end()) if m else (None, i)


def _parse_definitions(text):
    """({name: (n_args, default, body)}, text without the definitions)."""
    macros, out, pos = {}, [], 0
    for m in _DEFINITION.finditer(text):
        if m.start() < pos:
            continue
        command = m.group(0)
        name, i = _read_macro_name(text, m.end())
        if name is None:
            continue
        n_args, default, body, provide = 0, None, None, "provide" in command
        if command.startswith("\\DeclareMathOperator"):
            arg, i = _read_group(text, i)
            body = None if arg is None else "\\operatorname%s{%s}" % ("*" if command.endswith("*") else "", arg)
        elif command == "\\let":
            j = i
            while j < len(text) and text[j] in " \t=":
                j += 1
            target = _MACRO_NAME.match(text, j)
            if target:
                body, i = target.group(0), target.end()
        elif command.endswith("def"):
            params = _DEF_PARAMS.match(text, i).group(0)
            arg, end = _read_group(text, i + len(params))
            if arg is not None:
                body, i, n_args = arg, end, len(params) // 2
            else:   # delimited parameters (\def\a#1.{...}): drop the definition text up to its body
                arg, end = _read_group(text, text.find("{", i) if "{" in text[i:i + 200] else i)
                if arg is not None:
                    i = end
                body, name = None, None
        else:
            count, i = _read_group(text, i, "[")
            if count is not None:
                n_args = int(count) if count.strip().isdigit() else 0
                default, i = _read_group(text, i, "[")
            body, i = _read_group(text, i)
        if body is None and name is not None:
            continue
        out.append(text[pos:m.start()])
        pos = i
        if name and name not in _PROTECTED_MACROS and re.fullmatch(r"[a-zA-Z]+", name):
            if not (provide and name in macros):
                macros[name] = (min(n_args, 9), default, body)
    out.append(text[pos:])
    return macros, "".join(out)


def _read_argument(text, i):
    """(argument, end) of one macro argument: a {group}, a \\command or one character."""
    arg, end = _read_group(text, i)
    if arg is not None:
        return arg, end
    j = i
    while j < len(text) and text[j] in " \t\n":
        j += 1
    if j >= len(text) or text[j] in "}":
        return None, i
    m = _MACRO_NAME.match(text, j) if text[j] == "\\" else None
    return (m.group(0), m.end()) if m else (text[j], j + 1)


@_fallback
def _fb_expand_macros(text):
    """Expand the macros the document defines itself (see 2b); the definitions are removed."""
    macros, text = _parse_definitions(text)
    if not macros:
        return text
    pattern = re.compile(r"\\(" + "|".join(sorted(map(re.escape, macros), key=len, reverse=True))
                         + r")(?![a-zA-Z])")
    limit = 4 * len(text) + 1_000_000
    for _ in range(MAX_MACRO_PASSES):
        out, pos, changed = [], 0, False
        for m in pattern.finditer(text):
            if m.start() < pos:
                continue
            n_args, default, body = macros[m.group(1)]
            i, args = m.end(), []
            if default is not None:
                opt, i = _read_group(text, i, "[")
                args.append(default if opt is None else opt)
            while len(args) < n_args:
                arg, i2 = _read_argument(text, i)
                if arg is None:
                    break
                args.append(arg)
                i = i2
            if len(args) < n_args:
                continue
            if n_args == 0 and text.startswith("{}", i):   # \ours{}: the braces only end the name
                i += 2
            out.append(text[pos:m.start()])
            out.append(re.sub(r"#(\d)", lambda a: args[int(a.group(1)) - 1]
                              if 0 < int(a.group(1)) <= len(args) else "", body))
            pos, changed = i, True
        if not changed:
            break
        out.append(text[pos:])
        text = "".join(out)
        if len(text) > limit:
            break
    return text


MATH_ENVS = ("equation", "align", "alignat", "gather", "multline", "eqnarray", "flalign", "displaymath",
             "math", "dmath")
_MATH_SCAN = re.compile(r"\\begin\{(" + "|".join(MATH_ENVS) + r")(\*?)\}|\\.|\$", re.S)
_CLOSER_SCAN = re.compile(r"\\.|\$|\n[ \t]*\n", re.S)


def _math_close(text, i, closer, inline):
    """End index of the math opened before i ('$', '$$', '\\)' or '\\]'), or -1.

    Escapes (\\$, \\\\) are skipped; inline math cannot contain a blank line, as in TeX."""
    for m in _CLOSER_SCAN.finditer(text, i):
        token = m.group(0)
        if token[0] == "\n":
            if inline:
                return -1
        elif token == "$":
            if closer == "$":
                return m.end()
            if closer == "$$" and text.startswith("$", m.end()):
                return m.end() + 1
        elif token == closer:
            return m.end()
    return -1


def _math_spans(text):
    """[(start, end)] of the math in text, read left to right like TeX: $...$, $$...$$, \\(...\\), \\[...\\]
    and the math environments. An opening $ that is never closed (in the same paragraph) is a plain $."""
    spans, i = [], 0
    while True:
        m = _MATH_SCAN.search(text, i)
        if not m:
            return spans
        token, end = m.group(0), -1
        if m.group(1):
            close = f"\\end{{{m.group(1)}{m.group(2)}}}"
            k = text.find(close, m.end())
            end = k + len(close) if k >= 0 else -1
        elif token == "$":
            display = text.startswith("$", m.end())
            end = _math_close(text, m.end() + display, "$$" if display else "$", not display)
            if end < 0:
                i = m.end() + display
                continue
        elif token in ("\\(", "\\["):
            end = _math_close(text, m.end(), "\\)" if token == "\\(" else "\\]", token == "\\(")
        if end < 0:   # an escape such as \\ or \$, or an unclosed opening
            i = m.end()
            continue
        spans.append((m.start(), end))
        i = end


def _split_math(text):
    """[(is_math, segment), ...] covering text in order."""
    parts, pos = [], 0
    for start, end in _math_spans(text):
        if start > pos:
            parts.append((False, text[pos:start]))
        parts.append((True, text[start:end]))
        pos = end
    if pos < len(text):
        parts.append((False, text[pos:]))
    return parts


_ACCENT_MARKS = {"`": "\u0300", "'": "\u0301", "^": "\u0302", "~": "\u0303", "=": "\u0304", ".": "\u0307",
                 '"': "\u0308", "u": "\u0306", "r": "\u030a", "H": "\u030b", "v": "\u030c", "c": "\u0327",
                 "k": "\u0328"}
_ACCENT = re.compile(r"\\([`'^~=.\"])\s*(?:\{\s*(\\?[a-zA-Z])\s*\}|(\\?[a-zA-Z]))"
                     r"|\\([uvHrck])(?:\s*\{\s*(\\?[a-zA-Z])\s*\}|\s+([a-zA-Z]))")
_SPECIAL_LETTERS = {"ss": "ß", "o": "ø", "O": "Ø", "ae": "æ", "AE": "Æ", "oe": "œ", "OE": "Œ", "aa": "å",
                    "AA": "Å", "l": "ł", "L": "Ł", "i": "ı"}
_SPECIAL_LETTER = re.compile(r"\\(ss|ae|AE|oe|OE|aa|AA|o|O|l|L|i)(?![a-zA-Z])(?:\{\})?")


def _accent(match):
    mark = match.group(1) or match.group(4)
    base = next(g for g in (match.group(2), match.group(3), match.group(5), match.group(6)) if g)
    base = {"\\i": "i", "\\j": "j"}.get(base, base.lstrip("\\"))
    return unicodedata.normalize("NFC", base + _ACCENT_MARKS[mark])


def accents_to_unicode(text):
    text = _ACCENT.sub(_accent, text)
    return _SPECIAL_LETTER.sub(lambda m: _SPECIAL_LETTERS[m.group(1)], text)


_FLOAT_OPTION = re.compile(r"(\\begin\{(?:(?:figure|table|algorithm|sidewaysfigure|sidewaystable)\*?"
                           r"|wrapfigure|wraptable|subfigure)\})\s*\[[^\]]*\]")
_WRAP_ARGS = re.compile(r"(\\begin\{(?:wrapfigure|wraptable)\})\s*\{[^{}]*\}\s*(?:\[[^\]]*\])?\s*\{[^{}]*\}")
_NO_ARG_COMMANDS = (
    "centering", "raggedright", "raggedleft", "noindent", "indent", "small", "footnotesize", "scriptsize",
    "tiny", "large", "Large", "LARGE", "huge", "Huge", "normalsize", "bfseries", "itshape", "rmfamily",
    "sffamily", "ttfamily", "mdseries", "upshape", "slshape", "scshape", "normalfont", "bf", "it", "em", "sl",
    "sc", "rm", "sf", "tt", "toprule", "midrule", "bottomrule", "hline", "clearpage", "cleardoublepage",
    "newpage", "pagebreak", "nopagebreak", "smallskip", "medskip", "bigskip", "par", "maketitle",
    "tableofcontents", "listoffigures", "listoftables", "protect", "FloatBarrier", "sloppy", "hfill", "vfill",
    "hfil", "vfil", "null", "ignorespaces", "allowbreak", "justifying", "balance", "onecolumn", "twocolumn",
    "suppressfloats", "frenchspacing", "selectfont", "phantomsection", "xspace", "strut", "unskip", "relax",
    "vline", "endhead", "endfoot", "endfirsthead", "endlastfoot", "printindex", "IEEEpeerreviewmaketitle",
    "begingroup", "endgroup", "makeatletter", "makeatother", "leavevmode", "baselineskip", "today")
_NO_ARG = re.compile(r"\\(?:" + "|".join(_NO_ARG_COMMANDS) + r")(?![a-zA-Z])\*?(?:\[[^\]]*\])?[ \t]*")
_DROP_ONE = ("vspace", "hspace", "label", "thispagestyle", "pagestyle", "captionsetup", "bibliographystyle",
             "enlargethispage", "addvspace", "color", "pagecolor", "linespread", "cline", "cmidrule",
             "arrayrulecolor", "rowcolor", "cellcolor", "author", "affiliation", "affil", "address", "email",
             "institute", "date", "thanks", "orcid", "corref", "fntext", "cortext", "ead", "IEEEauthorblockN",
             "IEEEauthorblockA", "authornote", "institution", "country", "city", "postcode", "streetaddress",
             "acmConference", "acmYear", "acmDOI", "acmISBN", "copyrightyear", "setcopyright", "keywords",
             "ccsdesc", "titlerunning", "authorrunning", "footnotetext", "icmlkeywords", "newcolumntype",
             "graphicspath", "hypersetup", "lstset", "tikzset", "pgfplotsset", "sisetup", "renewenvironment",
             "fontencoding", "fontfamily", "fontseries", "fontshape", "theoremstyle", "pdfbookmark",
             "markright", "affiliations", "authors")
_DROP_TWO = ("setlength", "addtolength", "setcounter", "addtocounter", "icmlauthor", "icmlaffiliation",
             "icmlcorrespondingauthor", "fontsize", "definecolor", "newenvironment", "newtheorem", "markboth",
             "crefformat", "Crefformat")
_DROP_THREE = ("IfFileExists", "crefname", "Crefname", "crefrangeformat")
_UNWRAP = ("textbf", "textit", "emph", "texttt", "textsc", "textrm", "textsf", "textsl", "textup", "textmd",
           "textnormal", "underline", "uline", "mbox", "hbox", "makebox", "fbox", "framebox", "textcolor",
           "colorbox", "enquote", "MakeUppercase", "MakeLowercase", "uppercase", "lowercase", "hl", "sout",
           "textsuperscript", "textsubscript")
_TEXT_SUBS = [
    (re.compile(r"\\(?:ldots|dots|textellipsis)(?![a-zA-Z])(?:\{\})?"), "..."),
    (re.compile(r"---"), "\u2014"),
    (re.compile(r"--"), "\u2013"),
    (re.compile(r"``|''"), '"'),
    (re.compile(r"(?<!\\)\\[,;:! ]"), " "),
    (re.compile(r"(?<!\\)\\[@/]"), ""),
    (re.compile(r"\\%"), "%"),
    (re.compile(r"\\_"), "_"),
    (re.compile(r"\\#"), "#"),
    (re.compile(r"\\textasciitilde(?![a-zA-Z])(?:\{\})?"), "~"),
    (re.compile(r"(?<!\\)~"), " "),
    (re.compile(r"\\\\\*?(?:\s*\[[^\]]*\])?"), r"\\\\"),
    (re.compile(r"\\(?:newline|linebreak)(?![a-zA-Z])"), " "),
    (re.compile(r"\\caption\s*\[[^\]]*\]"), r"\\caption"),
    (re.compile(r"[ \t]+"), " "),
    (re.compile(r" ?\n ?"), "\n"),
    (re.compile(r"\n{3,}"), "\n\n"),
]


_SKIP_DIMENSION = re.compile(r"\\[vh]skip\s*-?[\d.]+\s*[a-z]{2}(?:\s*(?:plus|minus)\s*-?[\d.]+\s*[a-z]{2})*")
_LENGTH_ASSIGNMENT = re.compile(r"\\(?:setlength|addtolength)\s*\\[a-zA-Z@]+\s*\{[^{}]*\}")
_AT_BLOCK = re.compile(r"\\makeatletter.*?\\makeatother", re.S)


def _clean_commands(text):
    """Commands whose arguments may hold math: handled on the whole text, before math is set aside."""
    text = _AT_BLOCK.sub("", text)
    text = _parse_definitions(text)[1]   # definitions brought in by an expanded macro
    text = _FLOAT_OPTION.sub(r"\1", text)
    text = _WRAP_ARGS.sub(r"\1", text)
    text = _replace_commands(text, _DROP_THREE, 3)
    text = _replace_commands(text, _DROP_TWO, 2)
    text = _replace_commands(text, _DROP_ONE, 1)
    text = _replace_commands(text, ("resizebox",), 3, keep=3)
    text = _replace_commands(text, ("scalebox", "textcolor", "colorbox", "adjustbox"), 2, keep=2)
    for _ in range(3):   # nested wrappers: \textbf{\emph{x}}
        unwrapped = _replace_commands(text, _UNWRAP, 1, keep=1)
        if unwrapped == text:
            break
        text = unwrapped
    text = _LENGTH_ASSIGNMENT.sub("", text)
    text = _SKIP_DIMENSION.sub("", text)
    return _NO_ARG.sub("", text)


def _clean_text_segment(text):
    text = accents_to_unicode(text)
    for pattern, replacement in _TEXT_SUBS:
        text = pattern.sub(replacement, text)
    return text


@_fallback
def _fb_clean_latex(text):
    """Formatting-only LaTeX removed; structure, citations and math are kept (see 2b)."""
    return "".join(segment if is_math else _clean_text_segment(segment)
                   for is_math, segment in _split_math(_clean_commands(text)))


_ALIGNED = {"align": "aligned", "flalign": "aligned", "alignat": "aligned", "eqnarray": "aligned",
            "gather": "gathered", "multline": "gathered"}
_MATH_ENV = re.compile(r"\\begin\{(\w+)(\*?)\}(.*)\\end\{\1\2\}", re.S)
_FORMULA_NOISE = re.compile(r"\\(?:label|tag\*?)\s*\{[^{}]*\}|\\(?:nonumber|notag)(?![a-zA-Z])")


def _formula_body(body):
    body = _squash(_FORMULA_NOISE.sub("", body))
    return re.sub(r"(?:\\\\\s*)+$", "", body).strip()


def _normalized_math(span):
    """One math span in the single inline ($...$) or block (equation) format."""
    inner, block = None, True
    if span.startswith("$$"):
        body = span[2:-2]
    elif span.startswith("$") or span.startswith("\\("):
        body, block = span[1:-1] if span.startswith("$") else span[2:-2], False
    elif span.startswith("\\["):
        body = span[2:-2]
    else:
        m = _MATH_ENV.fullmatch(span)
        env, body = m.group(1), m.group(3)
        if env == "math":
            block = False
        if env == "alignat":
            body = re.sub(r"^\s*\{[^{}]*\}", "", body)
        inner = _ALIGNED.get(env)
    if not block:
        return "$" + _squash(body) + "$"
    body = _formula_body(body)
    if inner:
        body = f"\\begin{{{inner}}} {body} \\end{{{inner}}}"
    return f"\\begin{{equation}} {body} \\end{{equation}}"


@_fallback
def _fb_normalize_math(text):
    """Inline math as $...$, block math as \\begin{equation}...\\end{equation} (see 2b)."""
    text = re.sub(r"\\(?:begin|end)\{subequations\}", "", text)
    return "".join(_normalized_math(segment) if is_math else segment for is_math, segment in _split_math(text))


_ABBREVIATIONS = frozenset({
    "e.g", "i.e", "al", "fig", "figs", "eq", "eqs", "eqn", "eqns", "sec", "secs", "tab", "tabs", "ref", "refs",
    "no", "nos", "vs", "cf", "resp", "approx", "dr", "prof", "mr", "mrs", "ms", "st", "jr", "sr", "thm", "lem",
    "def", "prop", "cor", "ch", "chap", "app", "appx", "alg", "vol", "vols", "pp", "p", "ed", "eds", "viz",
    "ca", "inc", "ltd", "co", "dept", "univ", "w.r.t", "a.k.a", "wrt", "ex", "obs", "corr", "conj", "rem",
    "assump", "defn", "sect", "fn", "max", "min", "avg", "std", "est", "op", "u.s", "u.k", "ph.d", "m.sc",
    "b.sc", "et", "illus", "pt", "pts", "lit", "cit", "n.b", "q.v", "c.f", "mt", "mts", "jan", "feb", "mar",
    "apr", "jun", "jul", "aug", "sep", "sept", "oct", "nov", "dec"})
_PROTECT = re.compile(r"\x00(\d+)\x00")
_BOUNDARY = re.compile(r"\.([\"')\]\u201d\u2019]*)\s+")
_URL = re.compile(r"(?:https?://|www\.)[^\s{}]+")
_COMMAND_WITH_ARGS = re.compile(r"\\[a-zA-Z]+\*?(?:\s*\[[^\]]*\])*\s*\{")


def _protect(text):
    """Replace math, commands with their braces and URLs by \\x00n\\x00 tokens; returns (text, saved)."""
    saved = []

    def keep(segment):
        saved.append(segment)
        return f"\x00{len(saved) - 1}\x00"

    out = []
    for is_math, segment in _split_math(text):
        if is_math:
            out.append(keep(segment))
            continue
        pos = 0
        for m in _COMMAND_WITH_ARGS.finditer(segment):
            if m.start() < pos:
                continue
            end = m.end() - 1
            while end < len(segment) and segment[end] == "{":
                close = _group_end(segment, end)
                if close < 0:
                    break
                end = close
            if end == m.end() - 1:
                continue
            out.append(_URL.sub(lambda u: keep(u.group(0)), segment[pos:m.start()]))
            out.append(keep(segment[m.start():end]))
            pos = end
        out.append(_URL.sub(lambda u: keep(u.group(0)), segment[pos:]))
    return "".join(out), saved


def _ends_sentence(text, period, after):
    """True if the period at text[period] ends a sentence; `after` is the index of the next word."""
    word = re.search(r"([A-Za-z][A-Za-z.]*)$", text[:period])
    if word:
        token = word.group(1).lower()
        if token in _ABBREVIATIONS or token.rstrip(".") in _ABBREVIATIONS:
            return False
        if len(word.group(1)) == 1 and word.group(1).isupper():   # an initial: J. Smith
            return False
    nxt = text[after:after + 1]
    return bool(nxt) and not nxt.islower()


@_fallback
def _fb_split_sentences(text):
    """Sentences of one paragraph (see 2b)."""
    protected, saved = _protect(_squash(text))
    sentences, start = [], 0
    for m in _BOUNDARY.finditer(protected):
        if _ends_sentence(protected, m.start(), m.end()):
            sentences.append(protected[start:m.end()])
            start = m.end()
    sentences.append(protected[start:])
    restore = lambda s: _PROTECT.sub(lambda t: saved[int(t.group(1))], s)   # noqa: E731
    return [s for s in (restore(s).strip() for s in sentences) if s]

### 2c. Hierarchy (Member 2)

`build_tree` reads the document body (`\begin{document}` … `\end{document}`), with an abstract written
before it (or as `\abstract{...}`) moved in, and the title (`\title{...}`) as the content of the root.

- **Levels**: `\part`/`\chapter` (level 1, kind `chapter`), `\section` 2, `\subsection` 3, `\subsubsection`
  4, `\paragraph`/`\subparagraph` 5–6 (kind `paragraph`), starred or not. A heading is attached to the nearest
  open heading of a higher level, so a paper without `\chapter` has its sections under the document, and one
  with only `\subsection` has those under the document: the level is found from the document itself.
- **Excluded**: a section titled References / Bibliography (until the next heading of the same or higher
  level), `thebibliography`, `\bibliography{}`, `\printbibliography`. **Included**: Acknowledgements (also the
  `acknowledgements` environment and revtex's `\acknowledgments`), and everything after `\appendix`.
- **Leaves**: block formulas (the math environments of 2b, `$$`, `\[`) → `formula`; `figure`, `table`,
  `wrapfigure`, `algorithm`, a bare `tabular`, `tikzpicture`, `lstlisting`, `verbatim` → `figure`, with the
  cleaned environment as content (caption, graphics file, table cells); other text → paragraphs (blank lines)
  → `split_sentences` → `sentence`.
- **Lists**: `itemize`, `enumerate`, `description` (and the paralist variants) → an `itemize` node whose
  children are `item` nodes (content: the `\item[label]`), each holding the sentences, formulas and nested
  lists of that item.
- Any other environment (theorem, proof, center, minipage, ...) keeps its content in place; inside math
  (`$\begin{cases}...\end{cases}$`) an environment is part of the sentence. A paragraph with a graphic
  outside any figure environment (`\includegraphics`, `\captionof`) is a `figure`; `\bibitem` lines written
  without `thebibliography` are dropped like the reference list.

In [ ]:
_HEADINGS = {"part": ("chapter", 1), "chapter": ("chapter", 1), "section": ("section", 2),
             "subsection": ("subsection", 3), "subsubsection": ("subsubsection", 4),
             "paragraph": ("paragraph", 5), "subparagraph": ("paragraph", 6)}
LIST_ENVS = frozenset({"itemize", "enumerate", "description", "compactitem", "compactenum", "compactdesc",
                       "inparaenum", "asparaenum", "asparaitem", "inparaitem"})
FIGURE_ENVS = frozenset({"figure", "table", "wrapfigure", "wraptable", "sidewaysfigure", "sidewaystable",
                         "algorithm", "tabular", "tabularx", "tabulary", "longtable", "tikzpicture",
                         "lstlisting", "verbatim", "minted", "algorithmic", "algorithm2e", "SCfigure", "deluxetable",
                         "planotable", "listing", "subtable", "subfigure", "chart", "plate", "Verbatim"})
FORMULA_ENVS = frozenset({"equation", "align", "alignat", "gather", "multline", "eqnarray", "flalign",
                          "displaymath", "dmath"})
SKIPPED_ENVS = frozenset({"thebibliography", "titlepage", "filecontents"})
_ENV_ARGS = {"minipage": 1, "multicols": 1, "adjustbox": 1, "tcolorbox": 0, "spacing": 1, "list": 2,
             "subappendices": 0}
REFERENCE_TITLES = frozenset({"references", "reference", "bibliography", "literature cited", "works cited",
                              "cited literature", "references and notes", "literature", "references cited"})
_TREE_TOKEN = re.compile(
    r"\\(part|chapter|section|subsection|subsubsection|paragraph|subparagraph)(?![a-zA-Z])\*?"
    r"|\\begin\s*\{([^{}]+)\}|\\end\s*\{([^{}]+)\}|(?<!\\)\$\$|(?<!\\)\\\["
    r"|\\(appendix|bibliography|printbibliography|nocite|bibliographystyle|item|title)(?![a-zA-Z])")
_ACK_BEGIN = re.compile(r"\\begin\{(?:acknowledgements?|acknowledgments?|acks)\}")
_ACK_END = re.compile(r"\\end\{(?:acknowledgements?|acknowledgments?|acks)\}")
_ACK_COMMAND = re.compile(r"\\(?:acknowledgments?|acknowledgements?)(?![a-zA-Z])(?!\s*\{)")
_TITLE = re.compile(r"\\(?:title|icmltitle)(?![a-zA-Z])\*?\s*(?:\[[^\]]*\])?")
_ABSTRACT_ENV = re.compile(r"\\begin\{abstract\}.*?\\end\{abstract\}", re.S)
_LOOSE_BIBITEM = re.compile(r"\\bibitem\s*(?:\[[^\]]*\])?\s*\{[^{}]*\}.*?(?=\\bibitem|\n[ \t]*\n|\\(?:sub)*section"
                            r"|\\end\{|\\appendix|\Z)", re.S)


def _node(kind, content="", children=None):
    return {"kind": kind, "content": content, "children": children or []}


def _plain_title(title):
    title = _replace_commands(title, ("thanks", "footnote", "label", "footnotemark"), 1)
    title = _replace_commands(title, ("texorpdfstring",), 2, keep=1)
    return _squash(title.replace("\\\\", " ").replace("\\&", "&")).strip(" .")


def _find_env_end(text, env, start):
    """(start, end) of the \\end{env} that closes the environment opened before `start`."""
    pattern = re.compile(r"\\(begin|end)\s*\{" + re.escape(env) + r"\}")
    depth = 1
    for m in pattern.finditer(text, start):
        depth += 1 if m.group(1) == "begin" else -1
        if depth == 0:
            return m.start(), m.end()
    return len(text), len(text)


def _structural(match):
    """True for a token that is never inside math: a heading or a figure/list/formula/abstract environment."""
    env = (match.group(2) or match.group(3) or "").strip().rstrip("*")
    return bool(match.group(1)) or env in FORMULA_ENVS or env in FIGURE_ENVS or env in LIST_ENVS \
        or env in ("abstract",) or env in SKIPPED_ENVS


def _events(text):
    """The block of LaTeX as a flat list of events in document order:
    ('head', kind, level, title), ('text', s), ('formula', s), ('figure', s), ('list', inner),
    ('abstract', inner) and ('break',) (an environment boundary: a new paragraph).
    Any other token inside math ($\\begin{cases}...$) is part of the sentence, not an event."""
    spans = _math_spans(text)
    starts = [s for s, _ in spans]
    events, i, last = [], 0, 0
    while True:
        m = _TREE_TOKEN.search(text, i)
        if not m:
            break
        k = bisect.bisect_right(starts, m.start()) - 1
        if k >= 0 and spans[k][0] < m.start() < spans[k][1] and not _structural(m):
            i = spans[k][1]
            continue
        if text[last:m.start()].strip():
            events.append(("text", text[last:m.start()]))
        i = m.end()
        if m.group(1):
            kind, level = _HEADINGS[m.group(1)]
            _, j = _read_group(text, i, "[")
            title, j = _read_group(text, j)
            if title is not None:
                events.append(("head", kind, level, _plain_title(title)))
                i = j
        elif m.group(2):
            env = m.group(2).strip()
            base = env.rstrip("*")
            if base in FORMULA_ENVS or base in FIGURE_ENVS or base in LIST_ENVS or base == "abstract" \
                    or base in SKIPPED_ENVS:
                end_start, end_end = _find_env_end(text, env, i)
                if base in FORMULA_ENVS:
                    events.append(("formula", text[m.start():end_end]))
                elif base in FIGURE_ENVS:
                    events.append(("figure", text[m.start():end_end]))
                elif base in LIST_ENVS:
                    events.append(("list", text[i:end_start]))
                elif base == "abstract":
                    events.append(("abstract", text[i:end_start]))
                i = end_end
            else:   # keep the content of any other environment, drop its markers and arguments
                _, i = _read_group(text, i, "[")
                for _ in range(_ENV_ARGS.get(base, 0)):
                    _, i = _read_group(text, i)
                events.append(("break",))
        elif m.group(3):
            events.append(("break",))
        elif m.group(0) in ("$$", "\\["):
            close = "$$" if m.group(0) == "$$" else "\\]"
            k = text.find(close, i)
            if k >= 0:
                events.append(("formula", "\\begin{equation} " + text[i:k].strip() + " \\end{equation}"))
                i = k + len(close)
        else:   # \appendix, \bibliography{...}, a stray \item, \title{...} (already read) ...
            name = m.group(4)
            if name in ("bibliography", "nocite", "bibliographystyle", "title"):
                _, i = _read_group(text, i, "[")
                _, i = _read_group(text, i)
            elif name == "printbibliography":
                _, i = _read_group(text, i, "[")
            events.append(("break",))
        last = i
    if text[last:].strip():
        events.append(("text", text[last:]))
    return events


_ITEM_SCAN = re.compile(r"\\item(?![a-zA-Z])|\\(begin|end)\s*\{(?:" + "|".join(LIST_ENVS) + r")\*?\}")


def _split_items(inner):
    """[(label or None, text)] of the top-level \\item of a list body."""
    items, depth, start, label = [], 0, None, None
    for m in _ITEM_SCAN.finditer(inner):
        if m.group(1) == "begin":
            depth += 1
        elif m.group(1) == "end":
            depth -= 1
        elif depth == 0:
            if start is not None:
                items.append((label, inner[start:m.start()]))
            label, start = _read_group(inner, m.end(), "[")
    if start is not None:
        items.append((label, inner[start:]))
    return items


def _has_text(sentence):
    return bool(re.search(r"\w", re.sub(r"\\[a-zA-Z]+\*?", "", sentence)))


_LOOSE_FIGURE = re.compile(r"\\(?:includegraphics|captionof|includepdf|includesvg)(?![a-zA-Z])")


def _sentences(text):
    leaves = []
    for paragraph in re.split(r"\n[ \t]*\n", text):
        paragraph = re.sub(r"\\(?:item|appendix|maketitle|noindent|centering)(?![a-zA-Z])", " ", paragraph)
        if not paragraph.strip():
            continue
        if _LOOSE_FIGURE.search(paragraph):   # a graphic outside a figure environment (minipage, center)
            leaves.append(_node("figure", _squash(paragraph)))
            continue
        for sentence in split_sentences(paragraph):
            sentence = _squash(sentence)
            if _has_text(sentence):
                leaves.append(_node("sentence", sentence))
    return leaves


def _block_nodes(event):
    """Nodes of one non-heading event."""
    kind = event[0]
    if kind == "text":
        return _sentences(event[1])
    if kind in ("formula", "figure"):
        return [_node(kind, _squash(event[1]))]
    if kind == "list":
        items = [_node("item", _plain_title(label) if label else "", _children(body))
                 for label, body in _split_items(event[1])]
        return [_node("itemize", "", items)] if items else []
    if kind == "head":   # a heading inside a list item or an abstract: keep its title as text
        return _sentences(event[3])
    return []


def _children(text):
    return [node for event in _events(text) for node in _block_nodes(event)]


def _abstract_command_to_env(text):
    """\\abstract{...} (some classes) -> \\begin{abstract}...\\end{abstract}."""
    m = re.search(r"\\abstract(?![a-zA-Z])", text)
    inner, end = _read_group(text, m.end()) if m else (None, 0)
    if inner is None:
        return text
    return text[:m.start()] + "\\begin{abstract}" + inner + "\\end{abstract}" + text[end:]


@_fallback
def _fb_build_tree(text):
    """Document tree of one version (see 2c)."""
    m = _TITLE.search(text)
    title = _read_group(text, m.end())[0] if m else None
    text = _abstract_command_to_env(text)
    body_match = re.search(r"\\begin\s*\{document\}", text)
    if body_match:
        preamble, body = text[:body_match.start()], text[body_match.end():]
        end = re.search(r"\\end\s*\{document\}", body)
        body = body[:end.start()] if end else body
        abstract = _ABSTRACT_ENV.search(preamble)
        if abstract and "\\begin{abstract}" not in body:
            body = abstract.group(0) + "\n" + body
    else:
        body = re.sub(r"\\(?:documentclass|usepackage|RequirePackage)\s*(?:\[[^\]]*\])?\s*\{[^{}]*\}", "", text)
    body = _LOOSE_BIBITEM.sub("", body)   # a reference list written without thebibliography
    body = _ACK_BEGIN.sub(r"\\section*{Acknowledgements}", body)
    body = _ACK_END.sub("", body)
    body = _ACK_COMMAND.sub(r"\\section*{Acknowledgments}", body)

    root = _node("document", _plain_title(title) if title else "")
    stack = [(0, root)]
    skip_level = None   # inside a References section
    for event in _events(body):
        if event[0] == "head":
            _, kind, level, heading = event
            if skip_level is not None and level > skip_level:
                continue
            skip_level = None
            if heading.lower().strip(" :.") in REFERENCE_TITLES:
                skip_level = level
                continue
            while stack[-1][0] >= level:
                stack.pop()
            node = _node(kind, heading)
            stack[-1][1]["children"].append(node)
            stack.append((level, node))
        elif skip_level is not None:
            continue
        elif event[0] == "abstract":
            root["children"].append(_node("abstract", "Abstract", _children(event[1])))
        else:
            stack[-1][1]["children"].extend(_block_nodes(event))
    return root

### 2d. References (Member 3)

- `extract_bib` reads the `.bib` files named by `\bibliography{a,b}` / `\addbibresource{...}` (relative to the
  version folder or the root's folder, ignoring case; if none is found and the version has exactly one
  `.bib`, that one) and keeps the entries the paper cites (`\cite`, `\citep`, `\citet`, `\parencite`, ...;
  all of them with `\nocite{*}` or when no citation command is found). Entries that are not cited are skipped
  after reading their key, without parsing their fields: some papers ship the whole ACL Anthology (a 50 MB
  `.bib`, 86,000 entries), which took a minute per version when parsed in full. It adds every `\bibitem` of a
  `thebibliography` written in the `.tex`, converted to a `misc` entry: authors, title and venue from the
  `\newblock` parts if present, else from quotes or the leading author names; year, DOI, arXiv ID (as an
  `arxiv.org/abs` URL) and URL by regular expressions.
- Fields: `author` split at `and` (outside braces), `Last, First` turned into `First Last`; `year` an
  integer (also from a biblatex `date`); `doi` without the `https://doi.org/` prefix; `journal` is `journal`,
  else `journaltitle`, else `booktitle`; `url` is `url`, else the arXiv URL from `eprint`. Braces, accents
  and formatting commands are cleaned; `@string` abbreviations are substituted.
- `dedup_bib`: two entries are the same reference if they share the key, the DOI, the arXiv ID or the
  normalised title (letters and digits only, at least 16 characters), inside one version or across
  versions. The key used most often (then the earliest) is kept and the others map to it; fields are
  **unioned**: each field from the kept key's entry if it has one, else from the others; the longest author
  list; a specific entry type over `misc`.
- `rename_cites` rewrites the keys of every citation command through `key_map` (a key cited twice after
  renaming is kept once).

In [ ]:
_BIB_REFERENCE = re.compile(r"\\(?:bibliography|addbibresource|addglobalbib)\s*(?:\[[^\]]*\])?\s*\{([^{}]*)\}")
_NOCITE_ALL = re.compile(r"\\nocite\s*\{\s*\*\s*\}")
_CITE = re.compile(r"(\\[a-zA-Z]*cite[a-zA-Z]*\*?(?:\s*\[[^\]]*\]){0,2}\s*\{)([^{}]*)(\})")
_BIBITEM = re.compile(r"\\bibitem\s*(?:\[((?:[^\[\]]|\[[^\]]*\])*)\])?\s*\{([^{}]*)\}")
_ENTRY_START = re.compile(r"@\s*([a-zA-Z]+)\s*([{(])")
_FIELD_NAME = re.compile(r"\s*,?\s*([A-Za-z0-9_\-:.+]+)\s*=\s*")
_BARE_VALUE = re.compile(r"[^,#}\s)]+")
_ARXIV_ID = re.compile(r"(\d{4}\.\d{4,5}|[a-z\-]+(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?")
_YEAR = re.compile(r"\b(1[5-9]\d\d|20\d\d)\b")
BIB_FIELDS = ("citation_key", "entry_type", "title", "authors", "year", "doi", "journal", "url")


def _entry_end(text, open_index):
    """Index after the entry that opens with text[open_index] ('{' or '(')."""
    closing = "}" if text[open_index] == "{" else ")"
    depth, i = 0, open_index + 1
    while i < len(text):
        c = text[i]
        if c == "\\":
            i += 2
            continue
        if c == "{":
            depth += 1
        elif c == "}":
            if depth == 0 and closing == "}":
                return i + 1
            depth -= 1
        elif c == ")" and depth == 0 and closing == ")":
            return i + 1
        i += 1
    return len(text)


def _field_value(text, i, strings):
    """(value, end) of a field value: {..}, ".." or a number / @string name, joined with #."""
    parts = []
    while i < len(text):
        while i < len(text) and text[i] in " \t\n\r":
            i += 1
        if i >= len(text):
            break
        c = text[i]
        if c == "{":
            end = _group_end(text, i)
            end = len(text) if end < 0 else end
            parts.append(text[i + 1:end - 1])
        elif c == '"':
            depth, end = 0, i + 1
            while end < len(text) and not (text[end] == '"' and depth == 0 and text[end - 1] != "\\"):
                depth += {"{": 1, "}": -1}.get(text[end], 0)
                end += 1
            parts.append(text[i + 1:end])
            end += 1
        else:
            m = _BARE_VALUE.match(text, i)
            if not m:
                break
            word = m.group(0)
            parts.append(strings.get(word.lower(), word))
            end = i + len(word)
        i = end
        while i < len(text) and text[i] in " \t\n\r":
            i += 1
        if i < len(text) and text[i] == "#":
            i += 1
            continue
        break
    return "".join(parts), i


_ENTRY_KEY = re.compile(r"\s*([^,\s{}()=]+)\s*,")


def _parse_bibtex(text, keep=None):
    """[(entry_type, key, {field: raw value})] of a .bib file (@string substituted, @comment skipped).

    keep: only the entries with these keys are parsed; the others are skipped after reading their key
    (some papers ship the whole ACL Anthology, 50 MB and 86,000 entries, and cite 40 of them).
    """
    entries, strings, pos = [], {}, 0
    while True:
        m = _ENTRY_START.search(text, pos)
        if not m:
            break
        kind = m.group(1).lower()
        if keep is not None and kind not in ("string", "comment", "preamble"):
            key = _ENTRY_KEY.match(text, m.end())
            if key and key.group(1) not in keep:
                pos = key.end()   # jump to the next "@type{" without reading this entry
                continue
        end = _entry_end(text, m.end() - 1)
        body, pos = text[m.end():end - 1], end
        if kind in ("comment", "preamble"):
            continue
        if kind == "string":
            name = _FIELD_NAME.match(body)
            if name:
                strings[name.group(1).lower()], _ = _field_value(body, name.end(), strings)
            continue
        key, comma, rest = body.partition(",")
        if not comma or not key.strip() or "=" in key:
            continue
        fields, i = {}, 0
        while True:
            name = _FIELD_NAME.match(rest, i)
            if not name:
                break
            value, i = _field_value(rest, name.end(), strings)
            fields[name.group(1).lower()] = value
        entries.append((kind, key.strip(), fields))
    return entries


def _bib_text(value):
    """A BibTeX / \\bibitem value as plain text (math kept)."""
    if not value:
        return None
    out = []
    for is_math, segment in _split_math(value):
        if is_math:
            out.append(segment)
            continue
        segment = accents_to_unicode(segment)
        segment = _replace_commands(segment, ("url", "href"), 1, keep=1)
        segment = re.sub(r"\\(?:newblock|em|it|bf|sc|rm|tt|sl|sf)(?![a-zA-Z])", " ", segment)
        segment = re.sub(r"\\[a-zA-Z]+\*?", "", segment)
        segment = segment.replace("\\&", "&").replace("\\%", "%").replace("\\_", "_").replace("~", " ")
        segment = re.sub(r"\\(.)", r"\1", segment).replace("{", "").replace("}", "")
        segment = segment.replace("---", "\u2014").replace("--", "\u2013").replace("``", '"').replace("''", '"')
        out.append(segment)
    text = _squash("".join(out))
    return text or None


def _split_top(text, separator):
    """Split text at a regex separator, outside braces."""
    parts, depth, start, i = [], 0, 0, 0
    while i < len(text):
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
        elif depth == 0:
            m = separator.match(text, i)
            if m and m.end() > i:
                parts.append(text[start:i])
                start = i = m.end()
                continue
        i += 1
    parts.append(text[start:])
    return parts


_AND = re.compile(r"\s+and\s+", re.I)


def _authors(raw):
    names = []
    for part in _split_top(raw or "", _AND):
        name = _bib_text(part)
        if not name or name.lower() in ("others", "et al", "et al."):
            continue
        if "," in name:   # "Last, First" or "Last, Jr., First"
            pieces = [p.strip() for p in name.split(",")]
            name = " ".join(pieces[2:] + pieces[:1] + pieces[1:2]) if len(pieces) > 2 else f"{pieces[1]} {pieces[0]}"
        names.append(_squash(name))
    return names


def _year(value):
    m = _YEAR.search(value or "")
    return int(m.group(1)) if m else None


def _doi(value):
    if not value:
        return None
    value = re.sub(r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)", "", value.strip(), flags=re.I)
    return value.strip("{} ") or None


def _arxiv_url(*values):
    for value in values:
        m = re.search(r"arxiv[^0-9a-z]{0,12}(?:abs/)?" + _ARXIV_ID.pattern, value or "", re.I)
        if m:
            return f"https://arxiv.org/abs/{m.group(1)}"
    return None


def _bib_entry(kind, key, fields):
    """Contract dict of one .bib entry."""
    eprint = fields.get("eprint") or fields.get("arxivid") or ""
    prefix = (fields.get("archiveprefix") or fields.get("eprinttype") or "").lower()
    arxiv = (f"https://arxiv.org/abs/{eprint.strip()}" if eprint and ("arxiv" in prefix or _ARXIV_ID.fullmatch(eprint.strip()))
             else _arxiv_url(fields.get("journal"), fields.get("note"), fields.get("howpublished")))
    return {
        "citation_key": key, "entry_type": kind,
        "title": _bib_text(fields.get("title")),
        "authors": _authors(fields.get("author") or fields.get("editor")),
        "year": _year(fields.get("year") or fields.get("date")),
        "doi": _doi(fields.get("doi")),
        "journal": _bib_text(fields.get("journal") or fields.get("journaltitle") or fields.get("booktitle")),
        "url": _bib_text(fields.get("url")) or arxiv,
    }


_NAME = re.compile(r"(?:(?:[A-Z][a-z]?\.\s?-?)+\s*[A-Z][\w'\u2019\-]+(?:\s+[A-Z][\w'\-]+)?"
                   r"|[A-Z][\w'\-]+(?:\s+[A-Z][\w'\-]+)?,?\s+(?:[A-Z]\.\s?-?)+"
                   r"|[A-Z][a-z]+(?:\s+[A-Z]\.)?\s+(?:(?:van|von|de|der|da|di|le|la)\s+)*[A-Z][\w'\-]+)")
_INITIALS = re.compile(r"(?:[A-Z][a-z]?\.\s?-?)+")


def _venue(text):
    text = _bib_text(text) or ""
    text = re.sub(r"^(?:In:?\s+|in:?\s+)", "", text)
    text = re.split(r"[,(]|\d", text, maxsplit=1)[0]
    text = text.strip(" .,;:")
    return text or None


def _split_plain_reference(text):
    """(authors text, title, rest) of a \\bibitem without \\newblock."""
    quoted = re.search(r"``(.+?)''|\"(.+?)\"|\u201c(.+?)\u201d", text, re.S)
    if quoted:
        return text[:quoted.start()], next(g for g in quoted.groups() if g), text[quoted.end():]
    clean = _bib_text(text) or ""
    tokens = re.split(r"(,\s*|\s+and\s+|\s*&\s*)", clean)
    names, consumed = [], 0
    for k in range(0, len(tokens), 2):
        token = tokens[k].strip()
        if _NAME.fullmatch(token) or token.lower().startswith("et al") or \
                (names and _INITIALS.fullmatch(token)):
            names.append(token)
            consumed = sum(len(t) for t in tokens[:k + 2])
        else:
            break
    rest = clean[consumed:].strip(" ,.")
    pieces = re.split(r"(?<=[a-z\)\?])\.\s+", rest, maxsplit=1)
    if names and len(pieces) == 2 and len(pieces[0].split()) >= 3 and not re.search(r"\d", pieces[0]):
        return ", ".join(names), pieces[0], pieces[1]
    return ", ".join(names), None, rest


def _bibitem_entry(key, text):
    """Contract dict of one \\bibitem, as a misc entry."""
    url = re.search(r"\\url\s*\{([^{}]+)\}|\\href\s*\{([^{}]+)\}|(https?://[^\s{}]+)", text)
    doi = re.search(r"\b(10\.\d{4,9}/[^\s,;{}]+)", text)
    arxiv = _arxiv_url(text)
    blocks = re.split(r"\\newblock(?![a-zA-Z])", text)
    if len(blocks) >= 2:
        authors_text, title, rest = blocks[0], blocks[1], " ".join(blocks[2:])
    else:
        authors_text, title, rest = _split_plain_reference(text)
    without_ids = re.sub(r"\b10\.\d{4,9}/\S+|\d{4}\.\d{4,5}|https?://\S+", " ", text)
    authors_text = re.sub(r"\s+and\s+", ", ", _bib_text(authors_text) or "")
    authors = []
    for part in (p.strip(" .,;") for p in authors_text.split(",")):
        if not part or part.lower().startswith("et al"):
            continue
        if authors and _INITIALS.fullmatch(part + ".") and " " not in authors[-1]:
            authors[-1] = f"{part}. {authors[-1]}".replace("..", ".")
        else:
            authors.append(part)
    return {
        "citation_key": key, "entry_type": "misc",
        "title": (_bib_text(title) or "").strip(" .,") or None,
        "authors": [a for a in authors if re.search(r"[A-Za-z]", a)],
        "year": _year(without_ids),
        "doi": doi.group(1).rstrip(".") if doi else None,
        "journal": _venue(rest),
        "url": (next(g for g in url.groups() if g).rstrip(".") if url else None) or arxiv,
    }


def _bibitems(raw_text):
    """Entries of every \\bibitem in the text."""
    entries, matches = [], list(_BIBITEM.finditer(raw_text))
    for n, m in enumerate(matches):
        end = matches[n + 1].start() if n + 1 < len(matches) else len(raw_text)
        text = raw_text[m.end():end]
        close = text.find("\\end{thebibliography}")
        text = text[:close] if close >= 0 else text[:5000]
        if m.group(2).strip():
            entries.append(_bibitem_entry(m.group(2).strip(), text.strip()))
    return entries


def _bib_files(vdir, raw_text):
    """(names referenced, relative paths of the .bib files found) of a version."""
    files = _version_files(vdir)
    bibs = [name for name in files if name.lower().endswith(".bib")]
    lower = {name.lower(): name for name in bibs}
    root = find_root(vdir)
    try:
        root_folder = Path(root).resolve().parent.relative_to(Path(vdir).resolve()).as_posix() if root else "."
    except ValueError:
        root_folder = "."
    folders = dict.fromkeys((root_folder, "."))
    names = [n.strip() for m in _BIB_REFERENCE.finditer(raw_text) for n in m.group(1).split(",") if n.strip()]
    found = []
    for name in names:
        for folder in folders:
            hits = [h for c in (name, name + ".bib") if (h := lower.get(posixpath.normpath(posixpath.join(folder, c)).lower()))]
            if hits:
                found.append(hits[0])
                break
        else:
            base = PurePosixPath(name).name.lower()
            hits = [b for b in bibs if PurePosixPath(b).stem.lower() == base or PurePosixPath(b).name.lower() == base]
            if hits:
                found.append(hits[0])
    if names and not found and len(bibs) == 1:
        found = bibs
    return names, list(dict.fromkeys(found))


def _parsed_bib(vdir, name, keep=None):
    """Contract dicts of a .bib file of the version (only the keys in `keep`, if given)."""
    cache = _VERSION_CACHE[str(vdir)]["bib"]
    key = (name, None if keep is None else frozenset(keep))
    if key not in cache:
        cache[key] = [_bib_entry(*entry) for entry in _parse_bibtex(_version_files(vdir)[name], keep)]
    return cache[key]


_REAL_ENTRY = re.compile(r"@\s*(?!(?:string|comment|preamble)\b)[a-zA-Z]+\s*[{(]\s*[^,\s{}()=]+\s*,", re.I)


def _cited_keys(raw_text):
    return {k.strip() for m in _CITE.finditer(raw_text) for k in m.group(2).split(",") if k.strip()}


def _merge_entries(first, second):
    """Union of two entries of one reference: first's fields win, second fills the gaps."""
    merged = dict(first)
    for field in BIB_FIELDS:
        if field == "authors":
            if len(second.get("authors") or []) > len(merged.get("authors") or []):
                merged["authors"] = second["authors"]
        elif field == "entry_type":
            if merged.get("entry_type") in (None, "misc") and second.get("entry_type") not in (None, "misc"):
                merged["entry_type"] = second["entry_type"]
        elif merged.get(field) in (None, "") and second.get(field) not in (None, ""):
            merged[field] = second[field]
    return merged


@_fallback
def _fb_extract_bib(vdir, raw_text):
    """BibTeX entries of one version, from its .bib files and \\bibitem (see 2d)."""
    _, found = _bib_files(vdir, raw_text)
    cited = None if _NOCITE_ALL.search(raw_text) else _cited_keys(raw_text)
    entries = [entry for name in found for entry in _parsed_bib(vdir, name, cited or None)]
    by_key = {}
    for entry in entries + _bibitems(raw_text):
        key = entry["citation_key"]
        by_key[key] = _merge_entries(by_key[key], entry) if key in by_key else entry
    return list(by_key.values())


@_fallback
def _fb_bib_source(vdir, raw_text):
    """bib, bibitem, missing_bbl or none (see the contract)."""
    names, found = _bib_files(vdir, raw_text)
    if any(_REAL_ENTRY.search(_version_files(vdir)[name]) for name in found):
        return "bib"
    if _BIBITEM.search(raw_text):
        return "bibitem"
    return "missing_bbl" if names else "none"


def _normalized_title(title):
    text = unicodedata.normalize("NFKD", title or "").encode("ascii", "ignore").decode().lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text if len(text) >= 16 else None


@_fallback
def _fb_dedup_bib(by_version):
    """(entries, key_map): one entry per reference over all versions, fields unioned (see 2d)."""
    items = [entry for version in sorted(by_version) for entry in by_version[version] or []]
    parent = list(range(len(items)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    first_seen = {}
    for i, entry in enumerate(items):
        arxiv = _ARXIV_ID.search(entry.get("url") or "") if "arxiv.org" in (entry.get("url") or "") else None
        for signature in (("key", entry.get("citation_key")), ("doi", (entry.get("doi") or "").lower() or None),
                          ("arxiv", arxiv.group(1) if arxiv else None),
                          ("title", _normalized_title(entry.get("title")))):
            if signature[1] is None:
                continue
            if signature in first_seen:
                parent[find(i)] = find(first_seen[signature])
            else:
                first_seen[signature] = i
    groups = {}
    for i in range(len(items)):
        groups.setdefault(find(i), []).append(i)
    entries, key_map = [], {}
    for members in sorted(groups.values(), key=min):
        keys = Counter(items[i]["citation_key"] for i in members)
        order = {}
        for i in members:
            order.setdefault(items[i]["citation_key"], i)
        chosen = max(keys, key=lambda k: (keys[k], -order[k]))
        ranked = [i for i in members if items[i]["citation_key"] == chosen] + \
                 [i for i in members if items[i]["citation_key"] != chosen]
        merged = dict(items[ranked[0]])
        for i in ranked[1:]:
            merged = _merge_entries(merged, items[i])
        merged["citation_key"] = chosen
        entries.append(merged)
        key_map.update({key: chosen for key in keys if key != chosen})
    return entries, key_map


@_fallback
def _fb_rename_cites(text, key_map):
    """Citation commands with their keys renamed through key_map (see 2d)."""
    def rename(m):
        keys = []
        for key in (k.strip() for k in m.group(2).split(",")):
            key = key_map.get(key, key)
            if key and key not in keys:
                keys.append(key)
        return m.group(1) + ",".join(keys) + m.group(3)
    return _CITE.sub(rename, text)

### 2e. Use a member's function when it exists, the fallback otherwise

In [ ]:
IMPLEMENTATION = {}   # contract function -> the notebook that provides it, or "fallback"
for _notebook, _names in TEAM_NOTEBOOKS.items():
    for _name in _names:
        _current = globals().get(_name)
        if _current is None or getattr(_current, "_m2_fallback", False):
            globals()[_name] = globals()["_fb_" + _name]
            IMPLEMENTATION[_name] = "fallback"
        else:
            IMPLEMENTATION[_name] = _notebook


def _implementation_tag(notebook):
    used = {IMPLEMENTATION[name] == "fallback" for name in TEAM_NOTEBOOKS[notebook]}
    return "fallback" if used == {True} else "team" if used == {False} else "mixed"


PARSER_VERSION = PARSER_BASE + "|" + "|".join(f"{nb[:2]}:{_implementation_tag(nb)}" for nb in TEAM_NOTEBOOKS)

## 3. Element IDs and deduplication across versions

`_ElementStore` holds the elements of one paper. `add_version` walks one version's tree in document order,
gives every node its ID through the dedup keys of the introduction, and returns that version's edges, node
counts per kind and its tree with IDs (`[element_id, child, child, ...]`, which `15` writes to
`parsed.json`). The tree is checked first (`_checked_tree`), so a malformed tree from a member's notebook
fails before any ID is given, and the version is recorded as `error` without leaving half of it behind.

In [ ]:
LEAF_KINDS = frozenset({"sentence", "formula", "figure"})
KIND_PREFIX = {"document": "doc", "chapter": "ch", "section": "sec", "subsection": "ssec",
               "subsubsection": "sssec", "paragraph": "par", "abstract": "abs", "itemize": "list",
               "item": "item", "sentence": "s", "formula": "eq", "figure": "fig"}
KIND_ALIASES = {"table": "figure", "enumerate": "itemize", "description": "itemize", "list": "itemize",
                "equation": "formula", "math": "formula", "part": "chapter", "subparagraph": "paragraph"}
VERSION_STATUSES = ("ok", "empty_source", "no_root", "error")
ELEMENT_COLUMNS = ("element_id", "arxiv_id", "kind", "content")
HIERARCHY_COLUMNS = ("arxiv_id", "version", "element_id", "parent_id")
BIBTEX_COLUMNS = ("arxiv_id", "citation_key", "entry_type", "title", "authors", "year", "doi", "journal", "url")


def _checked_tree(node):
    """(kind, content, children) tuples of a tree, with kinds checked; raises ValueError if malformed."""
    if not isinstance(node, dict) or not isinstance(node.get("children", []), list):
        raise ValueError(f"tree node must be a dict with a children list, got {type(node).__name__}")
    kind = KIND_ALIASES.get(node.get("kind"), node.get("kind"))
    if kind not in KIND_PREFIX:
        raise ValueError(f"unknown node kind {node.get('kind')!r}")
    content = node.get("content")
    content = "" if content is None else _squash(str(content))
    return kind, content, [_checked_tree(child) for child in node.get("children", [])]


class _ElementStore:
    def __init__(self, folder_id):
        self.folder_id = folder_id
        self.root_id = f"{folder_id}#doc"
        self.ids = {}            # dedup key -> element_id
        self.counts = Counter()  # prefix -> last number used
        self.elements = []       # (element_id, kind, content) in order of first appearance

    def _id(self, kind, content, key):
        element_id = self.ids.get(key)
        if element_id is None:
            prefix = KIND_PREFIX[kind]
            self.counts[prefix] += 1
            element_id = f"{self.folder_id}#{prefix}{self.counts[prefix]}"
            self.ids[key] = element_id
            self.elements.append((element_id, kind, content))
        return element_id

    def add_version(self, version, tree):
        """(edges [(version, element_id, parent_id)], Counter of kinds, tree with IDs) of one version."""
        kind, content, children = _checked_tree(tree)
        if kind != "document":
            raise ValueError(f"the root of a tree must be a document node, got {kind!r}")
        if not any(element_id == self.root_id for element_id, _, _ in self.elements[:1]):
            self.elements.insert(0, (self.root_id, "document", content))
        edges, kinds, siblings = [], Counter({"document": 1}), Counter()

        def walk(nodes, parent_id):
            out = []
            for kind, content, children in nodes:
                if kind in LEAF_KINDS:
                    key = (kind, content)
                else:
                    k = siblings[(parent_id, kind, content)]
                    siblings[(parent_id, kind, content)] += 1
                    key = (kind, content, parent_id, k)
                element_id = self._id(kind, content, key)
                edges.append((version, element_id, parent_id))
                kinds[kind] += 1
                out.append([element_id, *walk(children, element_id)])
            return out

        return edges, kinds, [self.root_id, *walk(children, self.root_id)]

## 4. `parse_paper`

The steps of the introduction for one paper. A failure in one version (an exception in any function) makes
that version `error`, with the message, and the other versions go on; `15` also catches a failure of the
whole paper (e.g. in `dedup_bib`). `seconds` is the time spent on that version in every step.

Returned dict: `arxiv_id`, `folder_id`, `parser_version`, `versions` (one dict per version of
`metadata.json`: `version, status, bib_source, n_nodes, seconds, error`), `trees` (`{version: tree with IDs}`),
the rows `elements`, `hierarchy`, `bibtex` (dicts with the table columns), and `counts` (`nodes_by_kind`
before dedup, `bib_entries_before` dedup) for the statistics.

In [ ]:
def _error_text(err):
    return f"{type(err).__name__}: {err}"[:500]


def _has_tex(vdir):
    return any(path.suffix.lower() == ".tex" for path in Path(vdir).rglob("*") if path.is_file())


def _bibtex_row(arxiv_id, entry):
    """A bibtex row with the statement's types: authors a list of strings, year an int or None."""
    authors = entry.get("authors") or []
    if isinstance(authors, str):
        authors = [a.strip() for a in re.split(r"\s+and\s+", authors) if a.strip()]
    year = entry.get("year")
    if not isinstance(year, int) or isinstance(year, bool):
        year = _year(str(year)) if year not in (None, "") else None
    text = lambda field: None if entry.get(field) in (None, "") else str(entry[field])   # noqa: E731
    return {"arxiv_id": arxiv_id, "citation_key": str(entry.get("citation_key") or "").strip(),
            "entry_type": (text("entry_type") or "misc").lower(), "title": text("title"),
            "authors": [str(a) for a in authors], "year": year, "doi": text("doi"),
            "journal": text("journal"), "url": text("url")}


def parse_paper(pdir):
    """Rows of the three tables for one paper folder, every version of metadata.json (see section 4)."""
    pdir = Path(pdir)
    meta = read_json(pdir / "metadata.json")
    if not meta:
        raise FileNotFoundError(f"{pdir / 'metadata.json'} is missing")
    arxiv_id = split_version(meta["id"])[0]
    folder_id = to_folder_id(arxiv_id)
    _VERSION_CACHE.clear()
    versions, raws, by_version = [], {}, {}
    try:
        # Steps 1-2: gather, then the references on the raw text, version by version
        for version in range(1, int(meta.get("versions") or 0) + 1):
            vdir = pdir / f"v{version}"
            record = {"version": version, "status": "ok", "bib_source": "none", "n_nodes": 0,
                      "seconds": 0.0, "error": ""}
            versions.append(record)
            started = time.perf_counter()
            try:
                if not vdir.is_dir():
                    record.update(status="empty_source", error="no vN/ folder")
                elif not _has_tex(vdir):
                    record.update(status="empty_source", error="no .tex file")
                else:
                    raw = gather(vdir)
                    if raw is None:
                        record.update(status="no_root", error="no .tex file with \\documentclass")
                    else:
                        by_version[version] = list(extract_bib(vdir, raw) or [])
                        record["bib_source"] = bib_source(vdir, raw)
                        raws[version] = raw
            except Exception as err:
                record.update(status="error", error=_error_text(err))
                by_version.pop(version, None)
            record["seconds"] += time.perf_counter() - started

        # Step 3: one bibliography for the paper
        started = time.perf_counter()
        entries, key_map = dedup_bib(by_version) if by_version else ([], {})
        share = (time.perf_counter() - started) / max(len(raws), 1)

        # Steps 4-5: clean, rename citations, build the tree, give IDs
        store, edges, trees, nodes_by_kind = _ElementStore(folder_id), [], {}, Counter()
        for version, raw in raws.items():
            record = versions[version - 1]
            started = time.perf_counter()
            try:
                text = rename_cites(normalize_math(clean_latex(expand_macros(raw))), key_map or {})
                version_edges, kinds, tree = store.add_version(version, build_tree(text))
                edges += version_edges
                nodes_by_kind.update(kinds)
                trees[version] = tree
                record["n_nodes"] = sum(kinds.values())
            except Exception as err:
                record.update(status="error", error=_error_text(err))
            record["seconds"] += share + time.perf_counter() - started
    finally:
        _VERSION_CACHE.clear()
    for record in versions:
        record["seconds"] = round(record["seconds"], 4)

    # Step 6: rows
    bibtex = [_bibtex_row(arxiv_id, entry) for entry in entries]
    return {
        "arxiv_id": arxiv_id, "folder_id": folder_id, "parser_version": PARSER_VERSION,
        "versions": versions, "trees": trees,
        "elements": [dict(zip(ELEMENT_COLUMNS, (element_id, arxiv_id, kind, content)))
                     for element_id, kind, content in store.elements],
        "hierarchy": [dict(zip(HIERARCHY_COLUMNS, (arxiv_id, version, element_id, parent_id)))
                      for version, element_id, parent_id in edges],
        "bibtex": [row for row in bibtex if row["citation_key"]],
        "counts": {"nodes_by_kind": dict(nodes_by_kind),
                   "bib_entries_before": sum(len(e) for e in by_version.values())},
    }

## 5. Checks and the `parsed.json` format

`check_paper` returns the broken invariants of one result (an empty list when all hold): IDs well formed
(`<folder-id>#<prefix><n>`, prefix matching the kind) and unique; every `element_id` and `parent_id` of
`hierarchy` present in `elements` and never null; in each version, the only parent that is never a child is
`<folder-id>#doc`; every version of `metadata.json` listed once; versions that are not `ok` without rows.

`to_parsed_json` / `rows_from_parsed` convert a result to the `parsed.json` written by `15` and back. The
rows are stored column by column (`{"element_id": [...], ...}`), which keeps the file small.

In [ ]:
def check_paper(result, n_versions=None):
    """List of broken invariants of a parse_paper result (empty if none)."""
    problems, folder_id = [], result["folder_id"]
    ids = [row["element_id"] for row in result["elements"]]
    kinds = {row["element_id"]: row["kind"] for row in result["elements"]}
    pattern = re.compile(re.escape(folder_id) + r"#(doc|([a-z]+)(\d+))")
    if len(ids) != len(set(ids)):
        problems.append(f"{len(ids) - len(set(ids))} duplicate element_id")
    for element_id in ids:
        m = pattern.fullmatch(element_id)
        if not m or (m.group(1) == "doc") != (kinds[element_id] == "document") or \
                (m.group(2) and KIND_PREFIX.get(kinds[element_id]) != m.group(2)):
            problems.append(f"bad element_id {element_id!r} for kind {kinds[element_id]!r}")
    known = set(ids)
    by_version = {}
    for row in result["hierarchy"]:
        if row["parent_id"] is None:
            problems.append(f"null parent_id for {row['element_id']}")
        elif row["parent_id"] not in known:
            problems.append(f"parent_id {row['parent_id']} not in elements")
        if row["element_id"] not in known:
            problems.append(f"element_id {row['element_id']} not in elements")
        if row["element_id"] == f"{folder_id}#doc":
            problems.append(f"the document root is a child in v{row['version']}")
        by_version.setdefault(row["version"], []).append(row)
    for version, rows in by_version.items():
        roots = {r["parent_id"] for r in rows} - {r["element_id"] for r in rows}
        if roots != {f"{folder_id}#doc"}:
            problems.append(f"v{version}: roots {sorted(roots)}")
    listed = [v["version"] for v in result["versions"]]
    if n_versions is not None and listed != list(range(1, n_versions + 1)):
        problems.append(f"versions {listed} instead of 1..{n_versions}")
    for record in result["versions"]:
        if record["status"] not in VERSION_STATUSES:
            problems.append(f"v{record['version']}: unknown status {record['status']!r}")
        if record["status"] != "ok" and record["version"] in by_version:
            problems.append(f"v{record['version']} is {record['status']} but has hierarchy rows")
    return problems


def _columns(rows, columns):
    return {column: [row[column] for row in rows] for column in columns}


def to_parsed_json(result):
    """The parsed.json document of one paper."""
    return {
        "parser_version": result["parser_version"], "arxiv_id": result["arxiv_id"],
        "folder_id": result["folder_id"], "versions": result["versions"], "counts": result["counts"],
        "trees": {str(version): tree for version, tree in result["trees"].items()},
        "elements": _columns(result["elements"], ELEMENT_COLUMNS),
        "hierarchy": _columns(result["hierarchy"], HIERARCHY_COLUMNS),
        "bibtex": _columns(result["bibtex"], BIBTEX_COLUMNS),
    }


def rows_from_parsed(doc, table):
    """Rows (dicts) of one table ('elements', 'hierarchy' or 'bibtex') of a parsed.json document."""
    columns = doc[table]
    names = list(columns)
    return [dict(zip(names, values)) for values in zip(*(columns[name] for name in names))]

## 6. Offline tests

Run only when this notebook is opened on its own (`RUN_TEST = True`). They build small papers in a
temporary folder, so they need neither Drive nor the network. `ParsePaperTests` covers the cases agreed for
`parse_paper`; `FallbackTests` checks the fallback against the statement's rules (gathering, root choice,
levels, excluded and included sections, sentence splitting, math, `\bibitem`, bibliography dedup). The
fallback tests run only for the functions that still use the fallback.

In [ ]:
import tempfile
import unittest


def make_paper(root, arxiv_id, versions, n_versions=None):
    """A paper folder: versions = {n: {relative path: text}} (None: no vN/ folder)."""
    pdir = Path(root) / to_folder_id(arxiv_id)
    pdir.mkdir(parents=True, exist_ok=True)
    count = n_versions or max(versions)
    write_json(pdir / "metadata.json", {"id": arxiv_id, "status": "ok", "versions": count,
                                        "version_info": [{"version": v} for v in range(1, count + 1)]})
    for version, files in versions.items():
        if files is None:
            continue
        vdir = pdir / f"v{version}"
        vdir.mkdir(exist_ok=True)
        for name, text in files.items():
            (vdir / name).parent.mkdir(parents=True, exist_ok=True)
            (vdir / name).write_text(text, encoding="utf-8")
    return pdir


def tex(body, preamble=""):
    return "\\documentclass{article}\n" + preamble + "\n\\title{A Test Paper}\n\\begin{document}\n\\maketitle\n" \
        + body + "\n\\end{document}\n"


def by_content(result, kind=None):
    return {row["content"]: row["element_id"] for row in result["elements"] if kind in (None, row["kind"])}


class TempTestCase(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)

    def parse(self, versions, n_versions=None, arxiv_id="2609.32834"):
        result = parse_paper(make_paper(self.root, arxiv_id, versions, n_versions))
        self.assertEqual(check_paper(result, n_versions or max(versions)), [])
        return result


class ParsePaperTests(TempTestCase):
    def test_sentence_shared_by_two_versions_is_one_element_with_two_edges(self):
        body = "\\section{Intro}\nWe study citation matching. It is hard."
        result = self.parse({1: {"main.tex": tex(body)}, 2: {"main.tex": tex(body + " A new sentence.")}})
        sentence = by_content(result, "sentence")["We study citation matching."]
        self.assertEqual(sum(row["element_id"] == sentence for row in result["elements"]), 1)
        self.assertEqual(sorted(r["version"] for r in result["hierarchy"] if r["element_id"] == sentence), [1, 2])
        self.assertIn("A new sentence.", by_content(result, "sentence"))

    def test_renamed_citation_keys_give_one_element(self):
        bib = "@article{%s, title={Attention Is All You Need}, author={Vaswani, Ashish and Shazeer, Noam}, year={2017}}"
        v1 = {"main.tex": tex("We use transformers \\cite{a}.\n\\bibliography{refs}"), "refs.bib": bib % "a"}
        v2 = {"main.tex": tex("We use transformers \\cite{b}.\n\\bibliography{refs}"), "refs.bib": bib % "b"}
        result = self.parse({1: v1, 2: v2})
        sentences = [row for row in result["elements"] if row["kind"] == "sentence"]
        self.assertEqual([row["content"] for row in sentences], ["We use transformers \\cite{a}."])
        self.assertEqual(len(result["bibtex"]), 1)
        entry = result["bibtex"][0]
        self.assertEqual((entry["citation_key"], entry["year"], entry["authors"]),
                         ("a", 2017, ["Ashish Vaswani", "Noam Shazeer"]))
        self.assertEqual(result["counts"]["bib_entries_before"], 2)

    def test_rename_cites_with_a_given_key_map(self):
        self.assertEqual(rename_cites("see \\citep[p.~2]{b, c} and \\cite{a,b}", {"b": "a"}),
                         "see \\citep[p.~2]{a,c} and \\cite{a}")

    def test_same_subsection_title_under_different_sections_gets_different_ids(self):
        body = "\\section{Method}\n\\subsection{Setup}\nFirst setup.\n\\section{Experiments}\n\\subsection{Setup}\nSecond setup."
        result = self.parse({1: {"main.tex": tex(body)}})
        setups = [row["element_id"] for row in result["elements"] if row["content"] == "Setup"]
        self.assertEqual(len(setups), 2)
        parents = {r["element_id"]: r["parent_id"] for r in result["hierarchy"]}
        self.assertNotEqual(parents[setups[0]], parents[setups[1]])

    def test_two_untitled_lists_of_one_section_stay_apart(self):
        body = ("\\section{S}\n\\begin{itemize}\\item One.\\end{itemize}\nText between.\n"
                "\\begin{itemize}\\item Two.\\end{itemize}")
        result = self.parse({1: {"main.tex": tex(body)}, 2: {"main.tex": tex(body)}})
        lists = [row["element_id"] for row in result["elements"] if row["kind"] == "itemize"]
        self.assertEqual(len(lists), 2)   # and the same two IDs serve both versions
        self.assertEqual(sum(r["element_id"] in lists for r in result["hierarchy"]), 4)

    def test_version_without_folder_is_empty_source_without_rows(self):
        result = self.parse({1: {"main.tex": tex("Only one version has text.")}, 2: None}, n_versions=3)
        self.assertEqual([v["status"] for v in result["versions"]], ["ok", "empty_source", "empty_source"])
        self.assertEqual({r["version"] for r in result["hierarchy"]}, {1})

    def test_pdf_only_and_rootless_versions(self):
        result = self.parse({1: {}, 2: {"part.tex": "Just a fragment."}, 3: {"refs.bib": "@misc{a, title={x}}"}})
        self.assertEqual([v["status"] for v in result["versions"]], ["empty_source", "no_root", "empty_source"])
        self.assertEqual((result["elements"], result["hierarchy"]), ([], []))

    def test_ids_are_well_formed_unique_and_parents_exist(self):
        body = ("\\begin{abstract}We summarise.\\end{abstract}\n\\section{A}\nText $x=1$. More.\n"
                "\\begin{equation}a=b\\end{equation}\n\\begin{figure}[h]\\centering\\caption{Plot}\\end{figure}\n"
                "\\subsection{B}\\subsubsection{C}\\paragraph{D} Deep. \\begin{enumerate}\\item[i] x.\\end{enumerate}")
        result = self.parse({1: {"main.tex": tex(body)}, 2: {"main.tex": tex(body.replace("More.", "Changed."))}})
        self.assertEqual({row["kind"] for row in result["elements"]},
                         {"document", "abstract", "section", "subsection", "subsubsection", "paragraph",
                          "sentence", "formula", "figure", "itemize", "item"})
        self.assertTrue(all(r["parent_id"] for r in result["hierarchy"]))
        self.assertEqual(result["elements"][0], {"element_id": "2609-32834#doc", "arxiv_id": "2609.32834",
                                                 "kind": "document", "content": "A Test Paper"})

    def test_error_in_one_version_does_not_stop_the_others(self):
        original = globals()["build_tree"]
        calls = []

        def flaky(text):
            calls.append(text)
            if len(calls) == 1:
                raise RuntimeError("boom")
            return original(text)
        globals()["build_tree"] = flaky
        try:
            result = self.parse({1: {"main.tex": tex("One.")}, 2: {"main.tex": tex("Two.")}})
        finally:
            globals()["build_tree"] = original
        self.assertEqual([(v["status"], v["error"]) for v in result["versions"]],
                         [("error", "RuntimeError: boom"), ("ok", "")])

    def test_parsed_json_round_trip(self):
        result = self.parse({1: {"main.tex": tex("\\section{A} One. Two.")}})
        doc = json.loads(json.dumps(to_parsed_json(result)))
        for table in ("elements", "hierarchy", "bibtex"):
            self.assertEqual(rows_from_parsed(doc, table), result[table])
        self.assertEqual(doc["trees"]["1"][0], "2609-32834#doc")

    def test_fallback_is_used_when_no_member_notebook_exists(self):
        for name, source in IMPLEMENTATION.items():
            if source == "fallback":
                self.assertIs(globals()[name], globals()["_fb_" + name])
        self.assertIn("13.1|", PARSER_VERSION)


class FallbackTests(TempTestCase):
    def setUp(self):
        super().setUp()
        missing = [name for name in self.needs if IMPLEMENTATION[name] != "fallback"]
        if missing:
            self.skipTest(f"{', '.join(missing)} come from a member's notebook")

    needs = ("find_root", "gather", "build_tree", "expand_macros", "clean_latex", "normalize_math",
             "split_sentences", "extract_bib", "dedup_bib", "rename_cites", "bib_source")

    def test_comments_inputs_and_root_choice(self):
        files = {"main.tex": tex("\\input{sec/intro}\n% \\input{sec/old}\n\\include{sec/end}"),
                 "rebuttal.tex": tex("Rebuttal text."),
                 "sec/intro.tex": "Intro 50\\% done. % a comment\nSame line.\n\\input{sec/deep}",
                 "sec/deep.tex": "Deep \\input{main}text.", "sec/old.tex": "Old text.", "sec/end.tex": "End."}
        vdir = make_paper(self.root, "2609.1", {1: files}) / "v1"
        self.assertEqual(find_root(vdir).name, "main.tex")
        raw = gather(vdir)
        self.assertIn("Intro 50\\% done. \nSame line.", raw)
        self.assertIn("Deep", raw)
        self.assertIn("End.", raw)
        self.assertNotIn("Old text", raw)
        self.assertNotIn("Rebuttal", raw)

    def test_begin_document_inside_an_input_file(self):
        files = {"main.tex": "\\documentclass{article}\n\\input{preamble}\n\\input{paper}",
                 "preamble.tex": "\\newcommand{\\method}{TriDrive}", "paper.tex": "\\begin{document}\n\\section{Intro}\n\\method{} works.\n\\end{document}"}
        result = self.parse({1: files})
        self.assertIn("TriDrive works.", by_content(result, "sentence"))

    def test_old_mac_line_ends(self):
        # 2605.18061: lines end with \r only; the first comment must not swallow the whole file
        files = {"Solomon4.tex": tex("\\section{One}\rText here.").replace("\n", "\r").replace(
            "\\documentclass", "%&LaTeX\r\\documentclass")}
        result = self.parse({1: files}, arxiv_id="2605.18061")
        self.assertIn("Text here.", by_content(result, "sentence"))

    def test_documentclass_inside_an_included_preamble(self):
        # 2605.22101: main.tex starts with \include{preamble}, and preamble.tex holds \documentclass
        files = {"main.tex": "\\include{preamble}\n\\begin{document}\n\\section{Intro}\n\\G{} acts.\n\\end{document}",
                 "preamble.tex": "\\documentclass{article}\n\\newcommand{\\G}{The group}"}
        vdir = make_paper(self.root, "2605.22101", {1: files}) / "v1"
        self.assertEqual(find_root(vdir).name, "main.tex")
        result = self.parse({1: files}, arxiv_id="2605.22102")
        self.assertIn("The group acts.", by_content(result, "sentence"))

    def test_levels_references_appendix_and_acknowledgements(self):
        body = ("\\subsection{Only subsections}Text one.\n\\section*{Acknowledgements}Thanks.\n"
                "\\section*{References}\n\\begin{thebibliography}{1}\\bibitem{x} A. B. Title.\\end{thebibliography}\n"
                "\\appendix\n\\section{Proofs}\\subsection{Lemma}Proof text.")
        tree = build_tree(clean_latex(tex(body)))
        top = [(c["kind"], c["content"]) for c in tree["children"]]
        self.assertEqual(top, [("subsection", "Only subsections"), ("section", "Acknowledgements"),
                               ("section", "Proofs")])
        self.assertEqual(tree["children"][2]["children"][0]["content"], "Lemma")

    def test_environment_in_inline_math_and_loose_graphics(self):
        body = ("We define $f(x) = \\begin{cases} 1 & x>0 \\\\ 0 & \\text{else} \\end{cases}$ for all x. Next one.\n\n"
                "\\begin{minipage}{0.5\\textwidth}\\includegraphics{a.png}\\captionof{figure}{Loose.}\\end{minipage}\n\n"
                "\\bibitem{x} A. Author, Some title, 2020.")
        tree = build_tree(normalize_math(clean_latex(tex(body))))
        self.assertEqual([(c["kind"], c["content"][:20]) for c in tree["children"]],
                         [("sentence", "We define $f(x) = \\b"), ("sentence", "Next one."),
                          ("figure", "\\includegraphics{a.p")])

    def test_sentence_splitting_rules(self):
        text = ("We use e.g. a model with 3.5 layers, see Fig. 2 and Smith et al. (2020). The value $x = 1. 5$ "
                "is shown at \\url{https://a.b/c. d}. Next we cite \\cite{a.b, c}. J. Smith agrees. the end")
        self.assertEqual(split_sentences(text), [
            "We use e.g. a model with 3.5 layers, see Fig. 2 and Smith et al. (2020).",
            "The value $x = 1. 5$ is shown at \\url{https://a.b/c. d}.",
            "Next we cite \\cite{a.b, c}.", "J. Smith agrees. the end"])

    def test_cleanup_and_math_normalisation(self):
        text = ("\\begin{figure}[htpb]\\centering\\toprule a \\midrule\\end{figure} \\textbf{Bold} caf\\'e "
                "\\(a+b\\) $$ x \\label{e} $$ \\[y\\] \\begin{align*} p &= q \\\\ \\end{align*}")
        out = normalize_math(clean_latex(text))
        self.assertIn("\\begin{figure}a \\end{figure}", out)
        self.assertIn("Bold café $a+b$", out)
        self.assertEqual(out.count("\\begin{equation}"), 3)
        self.assertIn("\\begin{equation} \\begin{aligned} p &= q \\end{aligned} \\end{equation}", out)
        self.assertIn("\\begin{equation} x \\end{equation}", out)
        # $a$$b$ is two inline formulas, \\$ is a line break before math, \$ a dollar, a lone $ stays text
        tricky = "A $a$$b$ B \\\\$c$ costs \\$5 or $6.\n\nNext $d$ end"
        self.assertEqual([s for is_math, s in _split_math(tricky) if is_math], ["$a$", "$b$", "$c$", "$d$"])
        self.assertEqual(normalize_math(tricky), tricky)

    def test_macro_expansion(self):
        text = ("\\newcommand{\\R}{\\mathbb{R}}\\newcommand\\pair[2]{(#1,#2)}\\def\\ours{OurNet}"
                "\\DeclareMathOperator*{\\argmax}{arg\\,max}\\newcommand{\\opt}[1][x]{f(#1)}"
                "$\\R$ \\pair{a}{b} \\ours{} $\\argmax$ \\opt \\opt[y]")
        self.assertEqual(expand_macros(text).strip(),
                         "$\\mathbb{R}$ (a,b) OurNet $\\operatorname*{arg\\,max}$ f(x) f(y)")

    def test_bib_parsing_and_sources(self):
        bib = ('@string{nips = "NeurIPS"}\n@inproceedings{k1, title = {{BERT}: Pre-training}, '
               'author = "Devlin, Jacob and M{\\"u}ller, K.", booktitle = nips # " 2019", year = 2019, '
               'doi = {https://doi.org/10.1/abc}}\n@misc{k2, title={Unused}}\n'
               '@article{k3, title={X}, eprint={2101.00001}, archivePrefix={arXiv}, date={2021-01-02}}')
        files = {"main.tex": tex("A \\cite{k1} B \\citet{k3}.\n\\bibliography{refs}"), "refs.bib": bib}
        vdir = make_paper(self.root, "2609.2", {1: files}) / "v1"
        raw = gather(vdir)
        entries = {e["citation_key"]: e for e in extract_bib(vdir, raw)}
        self.assertEqual(sorted(entries), ["k1", "k3"])
        self.assertEqual(entries["k1"], {"citation_key": "k1", "entry_type": "inproceedings",
                                         "title": "BERT: Pre-training", "authors": ["Jacob Devlin", "K. Müller"],
                                         "year": 2019, "doi": "10.1/abc", "journal": "NeurIPS 2019", "url": None})
        self.assertEqual((entries["k3"]["year"], entries["k3"]["url"]), (2021, "https://arxiv.org/abs/2101.00001"))
        self.assertEqual(bib_source(vdir, raw), "bib")
        missing = make_paper(self.root, "2609.3", {1: {"main.tex": tex("\\cite{a}\\bibliography{gone}")}}) / "v1"
        self.assertEqual(bib_source(missing, gather(missing)), "missing_bbl")
        none = make_paper(self.root, "2609.4", {1: {"main.tex": tex("No references.")}}) / "v1"
        self.assertEqual(bib_source(none, gather(none)), "none")

    def test_bibitem_to_bibtex(self):
        raw = ("\\begin{thebibliography}{9}\n"
               "\\bibitem[{Vaswani et~al.(2017)}]{vas} A.~Vaswani, N.~Shazeer, and I.~Polosukhin.\n\\newblock "
               "Attention is all you need.\n\\newblock In \\emph{NeurIPS}, pages 1--11, 2017.\n"
               "\\bibitem{fro} A.M. Frolov, J. Chem. Phys. {\\bf 126}, 104302 (2007). doi:10.1063/1.2709880\n"
               "\\bibitem{q} J. Doe and R. Roe, ``A quoted title,'' arXiv:2101.00001, 2021.\n"
               "\\end{thebibliography}")
        entries = {e["citation_key"]: e for e in _bibitems(raw)}
        self.assertEqual((entries["vas"]["title"], entries["vas"]["authors"], entries["vas"]["year"],
                          entries["vas"]["journal"]),
                         ("Attention is all you need", ["A. Vaswani", "N. Shazeer", "I. Polosukhin"], 2017, "NeurIPS"))
        self.assertEqual((entries["fro"]["authors"], entries["fro"]["title"], entries["fro"]["year"],
                          entries["fro"]["doi"], entries["fro"]["journal"]),
                         (["A.M. Frolov"], None, 2007, "10.1063/1.2709880", "J. Chem. Phys"))
        self.assertEqual((entries["q"]["title"], entries["q"]["url"], entries["q"]["entry_type"]),
                         ("A quoted title", "https://arxiv.org/abs/2101.00001", "misc"))

    def test_bib_dedup_unions_fields_and_maps_keys(self):
        v1 = [{"citation_key": "a", "entry_type": "misc", "title": "Deep Residual Learning", "authors": ["K. He"],
               "year": 2016, "doi": None, "journal": None, "url": None}]
        v2 = [{"citation_key": "b", "entry_type": "inproceedings", "title": "Deep residual learning.",
               "authors": ["Kaiming He", "Xiangyu Zhang"], "year": None, "doi": "10.1109/CVPR.2016.90",
               "journal": "CVPR", "url": None},
              {"citation_key": "b2", "entry_type": "misc", "title": "Other", "authors": [], "year": None,
               "doi": "10.1109/cvpr.2016.90", "journal": None, "url": None}]
        entries, key_map = dedup_bib({1: v1, 2: v2})
        self.assertEqual(key_map, {"b": "a", "b2": "a"})
        self.assertEqual(entries, [{"citation_key": "a", "entry_type": "inproceedings",
                                    "title": "Deep Residual Learning", "authors": ["Kaiming He", "Xiangyu Zhang"],
                                    "year": 2016, "doi": "10.1109/CVPR.2016.90", "journal": "CVPR", "url": None}])


if RUN_TEST:
    _suite = unittest.TestSuite()
    for _case in (ParsePaperTests, FallbackTests):
        _suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(_case))
    _result = unittest.TextTestRunner(verbosity=2).run(_suite)
    if not _result.wasSuccessful():
        raise AssertionError("13_dedup_export offline tests failed")

In [ ]:
print(f"13 loaded | parser {PARSER_VERSION} | "
      f"fallback for: {', '.join(n for n, s in IMPLEMENTATION.items() if s == 'fallback') or 'none'}")